libraries

In [ ]:
from phytclust.main import PhytClust
from phytclust import plot_tree, plot_multiple_clusters
from phytclust import pairwise_distances
import io
import os
from Bio import Phylo
import sys
import numpy as np
import random
import matplotlib as mpl
import os
import datetime
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd

Figure 5A: Plotting Bird Tree

In [ ]:
from Bio import Phylo
import matplotlib.pyplot as plt
tree_string = (
"(((A:5, B:3)C1:6, (C:3, D:7)D1:4)A13:22, (((E:7, F:13)E12:5, G:6)B23:10, H:60)EF:35)GH:0;"
)
tree = Phylo.read(io.StringIO(tree_string), "newick")\

# Parse the Newick tree
# tree = Phylo.read("example_tree.newick", "newick")


In [ ]:
# from Bio import Phylo

# # Assume 'tree' is a Bio.Phylo tree, and you have clusters defined:
# clusters = [
#     ["A", "B"],  # Cluster 1
#     ["D", "C"],  # Cluster 2
#     ["E","F", "G", "H"],  # Cluster 3
# ]

# mrca_list = []
# for i, cluster in enumerate(clusters, start=1):
#     mrca = tree.common_ancestor(cluster)
#     mrca.name = f"Cluster_{i}"
#     mrca.clades = []
#     mrca_list.append([mrca])

# cluster_colors = ["red", "blue", "green"]

# # Create a map from clade to color
# clade_color_map = {}

# for i, cluster in enumerate(mrca_list):
#     color = cluster_colors[i]
#     mrca = tree.common_ancestor(cluster)

#     # Color the entire subtree under this MRCA
#     for descendant in mrca.find_clades():
#         clade_color_map[descendant] = color
#     if mrca in clade_color_map:
#         del clade_color_map[mrca]
# plot_tree(
#     tree,
#     clade_color_map=clade_color_map,
#     line_width=3,
#     height_scale=2,
#     width_scale=1,
#     hide_internal_nodes=True,
#     show_terminal_labels=False,
#     label_func=lambda x: "",
# )

In [ ]:
import os
import logging
import matplotlib.collections as mpcollections
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import logging
from collections import defaultdict
from typing import Any, Callable, Dict, List, Optional, Tuple, Union

logger = logging.getLogger(__name__)

COLORS = {
    "allele_a": mpl.colors.to_rgba("orange"),
    "allele_b": mpl.colors.to_rgba("teal"),
    "clonal": mpl.colors.to_rgba("lightgrey"),
    "normal": mpl.colors.to_rgba("dimgray"),
    "gain": mpl.colors.to_rgba("red"),
    "wgd": mpl.colors.to_rgba("green"),
    "loss": mpl.colors.to_rgba("blue"),
    "chr_label": mpl.colors.to_rgba("grey"),
    "vlines": "#1f77b4",
    "marker_internal": "#1f77b4",
    "marker_terminal": "black",
    "marker_normal": "green",
    "summary_label": "grey",
    "background": "white",
    "background_hatch": "lightgray",
    "patch_background": "white",
}
LINEWIDTHS = {
    "copy_numbers": 2,
    "chr_boundary": 1,
    "segment_boundary": 0.5,
}
ALPHAS = {
    "patches": 0.15,
    "patches_wgd": 0.3,
    "clonal": 0.3,
}
SIZES = {
    "tree_marker": 40,
    "ylabel_font": 8,
    "ylabel_tick": 6,
    "xlabel_font": 10,
    "xlabel_tick": 8,
    "chr_label": 8,
}


def plot_tree(
    input_tree: Any,
    label_func: Optional[Callable[[Any], str]] = None,
    title: str = "",
    ax: Optional[plt.Axes] = None,
    output_name: Optional[str] = None,
    outgroup: Optional[str] = None,
    width_scale: float = 1,
    height_scale: float = 1,
    show_terminal_labels: bool = False,
    show_branch_lengths: bool = True,
    show_branch_support: bool = False,
    show_events: bool = False,
    branch_labels: Optional[Union[Dict[Any, str], Callable[[Any], str]]] = None,
    label_colors: Optional[Union[Dict[str, str], Callable[[str], str]]] = None,
    hide_internal_nodes: bool = False,
    marker_size: Optional[int] = None,
    line_width: Optional[float] = None,
    clade_color_map: Optional[Dict[Any, str]] = None,  # <--- New parameter
    **kwargs: Any,
) -> plt.Figure:
    """Plot the given tree using matplotlib.

    ...
    Added: clade_color_map : optional dict of {clade: color_str}
        If provided, branches leading to the given clade will be drawn in the specified color.
    """
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        try:
            import pylab as plt
        except ImportError:
            raise MEDICCPlotError(
                "Install matplotlib or pylab if you want to use draw."
            ) from None

    import matplotlib.collections as mpcollections

    marker_size = marker_size or SIZES["tree_marker"]
    line_width = line_width or LINEWIDTHS.get("segment_boundary", 1.0)
    # label_func = label_func or (lambda x: x.name if hasattr(x, "name") else x)
    label_func = label_func or (lambda x: "" if hasattr(x, "name") else "")

    horizontal_lines = []
    vertical_lines = []
    horizontal_colors = []
    vertical_colors = []
    horizontal_lws = []
    vertical_lws = []

    marker_x = []
    marker_y = []
    marker_sizes = []
    marker_colors = []

    text_x = []
    text_y = []
    texts = []
    text_colors = []

    if ax is None:
        nsamp = len(list(input_tree.find_clades()))
        plot_height = height_scale * nsamp * 0.25
        max_leaf_to_root_distances = np.max(
            [
                np.sum([x.branch_length for x in input_tree.get_path(leaf)])
                for leaf in input_tree.get_terminals()
            ]
        )
        # plot_width = 5 + np.max(
        #     [0, width_scale * np.log10(max_leaf_to_root_distances / 100) * 5]
        # )
        plot_width = 5 + np.max(
            [0, width_scale  * 0.5]
        )
        # maximum figure size is 250x250 inches
        fig, ax = plt.subplots(figsize=(min(250, plot_width), min(250, plot_height)))

    get_label_color = lambda label: "black"
    if label_colors is not None:
        get_label_color = (
            label_colors
            if callable(label_colors)
            else lambda label: label_colors.get(label, "black")
        )
    else:
        clade_colors = {}
        for clade in input_tree.find_clades():
            sample = clade.name
            if sample is None:
                continue
            is_terminal = clade.is_terminal()
            clade_colors[sample] = (
                COLORS["marker_terminal"] if is_terminal else COLORS["marker_internal"]
            )
            if sample == outgroup:
                clade_colors[sample] = COLORS["marker_normal"]
        get_label_color = lambda label: clade_colors.get(label, "black")

    marker_size = marker_size if marker_size is not None else SIZES["tree_marker"]
    marker_func = lambda x: (
        (marker_size, get_label_color(x.name)) if x.name is not None else None
    )

    ax.axes.get_yaxis().set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True, prune=None))
    ax.xaxis.set_tick_params(labelsize=SIZES["xlabel_tick"])
    ax.xaxis.label.set_size(SIZES["xlabel_font"])
    ax.set_title(
        title,
        x=0.01,
        y=1.0,
        ha="left",
        va="bottom",
        fontweight="bold",
        fontsize=16,
        zorder=10,
    )
    x_posns = _get_x_positions(input_tree)
    y_posns = _get_y_positions(
        input_tree, adjust=not hide_internal_nodes, outgroup=outgroup
    )
    xmax = max(x_posns.values())
    ax.set_xlim(-0.05 * xmax, 1.05 * xmax)
    top_margin = 0.5
    ymax = max(y_posns.values()) + top_margin
    ax.set_ylim(ymax, -0.5)
    ax_scale = ax.get_xlim()[1] - ax.get_xlim()[0]

    def value_to_str(value):
        if value is None or value == 0:
            return None
        return str(int(value)) if int(value) == value else str(value)

    if not branch_labels:
        if show_branch_lengths:

            def format_branch_label(x):
                return (
                    value_to_str(np.round(x.branch_length, 1))
                    if x.name != "root" and x.name is not None
                    else None
                )

        else:

            def format_branch_label(clade):
                return None

    elif isinstance(branch_labels, dict):

        def format_branch_label(clade):
            return branch_labels.get(clade)

    else:
        if not callable(branch_labels):
            raise TypeError(
                "branch_labels must be either a dict or a callable (function)"
            )

        def format_branch_label(clade):
            return value_to_str(branch_labels(clade))

    if show_branch_support:

        def format_support_value(clade):
            if clade.name == "root" or clade.name is None:
                return None
            try:
                confidences = clade.confidences
            except AttributeError:
                pass
            else:
                return "/".join(value_to_str(cnf.value) for cnf in confidences)
            return (
                value_to_str(clade.confidence) if clade.confidence is not None else None
            )

    def draw_clade_lines(
        use_linecollection: bool = False,
        orientation: str = "horizontal",
        y_here: float = 0,
        x_start: float = 0,
        x_here: float = 0,
        y_bot: float = 0,
        y_top: float = 0,
        color: str = "black",
        lw: float = 0.1,
    ) -> None:
        if use_linecollection and orientation == "horizontal":
            horizontal_lines.append([(x_start, y_here), (x_here, y_here)])
            horizontal_colors.append(color)
            horizontal_lws.append(lw)
        elif use_linecollection and orientation == "vertical":
            vertical_lines.append([(x_here, y_bot), (x_here, y_top)])
            vertical_colors.append(color)
            vertical_lws.append(lw)

    def draw_clade(clade: Any, x_start: float, default_color: str, lw: float) -> None:
        x_here = x_posns[clade]
        y_here = y_posns[clade]

        # Determine line color for this clade (horizontal line to this node)
        line_color = (
            clade_color_map.get(clade, default_color) if clade_color_map else default_color
        )

        if hasattr(clade, "color") and clade.color is not None:
            line_color = clade.color.to_hex()
        if hasattr(clade, "width") and clade.width is not None:
            lw = clade.width * plt.rcParams["lines.linewidth"]

        # Draw horizontal line to this clade
        draw_clade_lines(
            use_linecollection=True,
            orientation="horizontal",
            y_here=y_here,
            x_start=x_start,
            x_here=x_here,
            color=line_color,
            lw=lw,
        )

        if marker_func is not None:
            marker = marker_func(clade)
            if (
                marker is not None
                and clade is not None
                and not (hide_internal_nodes and not clade.is_terminal())
            ):
                m_size, m_color = marker
                marker_x.append(x_here)
                marker_y.append(y_here)
                marker_sizes.append(m_size)
                marker_colors.append(m_color)

        label = label_func(str(clade.name))
        if label not in (None, clade.__class__.__name__) and not (
            hide_internal_nodes and not clade.is_terminal()
        ):
            text_x.append(x_here + min(0.02 * ax_scale, 1))
            text_y.append(y_here)
            texts.append(f" {label}")
            text_colors.append(get_label_color(label))

        if clade.clades:
            y_top_local = y_posns[clade.clades[0]]
            y_bot_local = y_posns[clade.clades[-1]]

            # Determine vertical line color based on children's colors
            child_colors = []
            for child in clade.clades:
                # Child color is looked up; if not found, fallback to parent's line_color
                child_line_color = (
                    clade_color_map.get(child, line_color)
                    if clade_color_map
                    else line_color
                )
                child_colors.append(child_line_color)

            # If all children share the same color, use that for the vertical line
            if len(set(child_colors)) == 1:
                vertical_line_color = child_colors[0]
            else:
                vertical_line_color = line_color

            # Draw vertical line with computed vertical_line_color
            draw_clade_lines(
                use_linecollection=True,
                orientation="vertical",
                x_here=x_here,
                y_bot=y_bot_local,
                y_top=y_top_local,
                color=vertical_line_color,
                lw=lw,
            )

            # Recurse into children, passing the parent's line_color as default
            for child in clade:
                draw_clade(child, x_here, line_color, lw)

    line_width = line_width if line_width is not None else plt.rcParams["lines.linewidth"]
    draw_clade(input_tree.root, 0, "k", line_width)

    if horizontal_lines:
        h_linecollection = mpcollections.LineCollection(
            horizontal_lines,
            colors=horizontal_colors,
            linewidths=horizontal_lws,
        )
        ax.add_collection(h_linecollection)

    if vertical_lines:
        v_linecollection = mpcollections.LineCollection(
            vertical_lines,
            colors=vertical_colors,
            linewidths=vertical_lws,
        )
        ax.add_collection(v_linecollection)

    if marker_x:
        ax.scatter(marker_x, marker_y, s=marker_sizes, c=marker_colors, zorder=3)

    for x, y, text, color in zip(text_x, text_y, texts, text_colors):
        ax.text(x, y, text, verticalalignment="center", color=color)

    ax.set_xlabel("branch length")
    ax.set_ylabel("taxa")

    for key, value in kwargs.items():
        try:
            list(value)
        except TypeError:
            raise ValueError(
                f'Keyword argument "{key}={value}" is not in the correct format.'
            ) from None

        if isinstance(value, dict):
            getattr(plt, str(key))(**dict(value))
        elif not (isinstance(value[0], tuple)):
            getattr(plt, str(key))(*value)
        elif isinstance(value[0], tuple):
            getattr(plt, str(key))(*value[0], **dict(value[1]))

    if output_name is not None:
        plt.savefig(output_name + ".png", bbox_inches="tight")

    return plt.gcf()


def _get_x_positions(tree: Any) -> Dict[Any, float]:
    """Create a mapping of each clade to its horizontal position.
    Dict of {clade: x-coord}
    """
    depths = tree.depths()
    # If there are no branch lengths, assume unit branch lengths
    if not max(depths.values()):
        depths = tree.depths(unit_branch_lengths=True)
    return depths


# def _get_y_positions(
#     tree: Any, adjust: bool = False, outgroup: str = "outgroup"
# ) -> Dict[Any, float]:
#     """Create a mapping of each clade to its vertical position.
#         Dict of {clade: y-coord}.
#     Coordinates are negative, and integers for tips.
#     """
#     maxheight = tree.count_terminals()
#     heights = {
#         tip: maxheight - 1 - i
#         for i, tip in enumerate(
#             reversed(
#                 [
#                     x
#                     for x in tree.get_terminals()
#                     if outgroup is None or x.name != outgroup
#                 ]
#             )
#         )
#     }
#     if outgroup is not None:
#         normal_clades = list(tree.find_clades(outgroup))
#         if not normal_clades:
#             raise PlotError(f"Normal clade {outgroup} not found in tree")
#         heights[normal_clades[0]] = maxheight

#     # Internal nodes: place at midpoint of children
#     def calc_row(clade: Any) -> None:
#         for subclade in clade:
#             if subclade not in heights:
#                 calc_row(subclade)
#         # Closure over heights
#         heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0

#     if tree.root.clades:
#         calc_row(tree.root)

#     if adjust:
#         pos = pd.DataFrame(
#             [(clade, val) for clade, val in heights.items()], columns=["clade", "pos"]
#         ).sort_values("pos")
#         pos["newpos"] = 0
#         count = 0
#         for i in pos.index:
#             if pos.loc[i, "clade"] != tree.root:
#                 count += 1
#             pos.loc[i, "newpos"] = count

#         pos.set_index("clade", inplace=True)
#         heights = pos.to_dict()["newpos"]

#     return heights


def _get_y_positions(
    tree: Any, adjust: bool = False, outgroup: str = "outgroup"
) -> Dict[Any, float]:
    """Create a mapping of each clade to its vertical position.
        Dict of {clade: y-coord}.
    Coordinates are negative, and integers for tips.
    """
    maxheight = tree.count_terminals()
    leaves = [x for x in tree.get_terminals() if outgroup is None or x.name != outgroup]
    heights = {tip: maxheight - 1 - i for i, tip in enumerate(reversed(leaves))}

    # heights = {
    #     tip: maxheight - 1 - i
    #     for i, tip in enumerate(
    #         reversed(
    #             [
    #                 x
    #                 for x in tree.get_terminals()
    #                 if outgroup is None or x.name != outgroup
    #             ]
    #         )
    #     )
    # }
    if outgroup is not None:
        normal_clades = list(tree.find_clades(outgroup))
        if not normal_clades:
            raise PlotError(f"Normal clade {outgroup} not found in tree")
        heights[normal_clades[0]] = maxheight

    # Internal nodes: place at midpoint of children
    def calc_row(clade: Any) -> None:
        for subclade in clade:
            if subclade not in heights:
                calc_row(subclade)
        if clade.clades:
            # The clade's position is the midpoint of its first and last child's position
            first_child = clade.clades[0]
            last_child = clade.clades[-1]
            heights[clade] = (heights[first_child] + heights[last_child]) / 2.0

    if tree.root.clades:
        calc_row(tree.root)

    if adjust:
        sorted_heights = sorted(heights.items(), key=lambda x: x[1])
        new_heights = {}
        count = 0
        for clade, _ in sorted_heights:
            # Keep the root at the initial reference, start counting from there
            if clade != tree.root:
                count += 1
            new_heights[clade] = count
        heights = new_heights

    return heights

In [ ]:
plot_tree(
    tree,
    marker_size=12,
    hide_internal_nodes=True,
    line_width=3,
    show_terminal_labels=False,
    label_func=lambda x: ""
)

In [ ]:
from matplotlib.patches import Rectangle


def plot_tree(
    input_tree: Any,
    label_func: Optional[Callable[[Any], str]] = None,
    title: str = "",
    ax: Optional[plt.Axes] = None,
    output_name: Optional[str] = None,
    outgroup: Optional[str] = None,
    width_scale: float = 1,
    height_scale: float = 1,
    show_terminal_labels: bool = False,
    show_branch_lengths: bool = True,
    show_branch_support: bool = False,
    show_events: bool = False,
    branch_labels: Optional[Union[Dict[Any, str], Callable[[Any], str]]] = None,
    label_colors: Optional[Union[Dict[str, str], Callable[[str], str]]] = None,
    hide_internal_nodes: bool = False,
    marker_size: Optional[int] = None,
    line_width: Optional[float] = None,
    clade_color_map: Optional[Dict[Any, str]] = None,
    cluster_mrcas: Optional[
        List[Any]
    ] = None,  # New parameter: List of MRCA clades for clusters
    cluster_labels: Optional[
        List[str]
    ] = None,  # New parameter: Labels for these clusters
    **kwargs: Any,
) -> plt.Figure:
    """
    Plot the given tree using matplotlib, and optionally highlight cluster MRCAs.
    cluster_mrcas: list of clade objects representing MRCAs of clusters you want to highlight
    cluster_labels: list of strings for labeling these clusters. Should match length of cluster_mrcas.
    """
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        try:
            import pylab as plt
        except ImportError:
            raise MEDICCPlotError(
                "Install matplotlib or pylab if you want to use draw."
            ) from None

    import matplotlib.collections as mpcollections

    marker_size = marker_size or SIZES["tree_marker"]
    line_width = line_width or LINEWIDTHS.get("segment_boundary", 1.0)
    label_func = label_func or (lambda x: x.name if hasattr(x, "name") else x)

    horizontal_lines = []
    vertical_lines = []
    horizontal_colors = []
    vertical_colors = []
    horizontal_lws = []
    vertical_lws = []

    marker_x = []
    marker_y = []
    marker_sizes = []
    marker_colors = []

    text_x = []
    text_y = []
    texts = []
    text_colors = []

    if ax is None:
        nsamp = len(list(input_tree.find_clades()))
        plot_height = height_scale * nsamp * 0.25
        max_leaf_to_root_distances = np.max(
            [
                np.sum([x.branch_length for x in input_tree.get_path(leaf)])
                for leaf in input_tree.get_terminals()
            ]
        )
        plot_width = 5 + np.max([0, width_scale * 0.5])
        fig, ax = plt.subplots(figsize=(min(250, plot_width), min(250, plot_height)))

    get_label_color = lambda label: "black"
    if label_colors is not None:
        get_label_color = (
            label_colors
            if callable(label_colors)
            else lambda label: label_colors.get(label, "black")
        )
    else:
        clade_colors = {}
        for clade in input_tree.find_clades():
            sample = clade.name
            if sample is None:
                continue
            is_terminal = clade.is_terminal()
            clade_colors[sample] = (
                COLORS["marker_terminal"] if is_terminal else COLORS["marker_internal"]
            )
            if sample == outgroup:
                clade_colors[sample] = COLORS["marker_normal"]
        get_label_color = lambda label: clade_colors.get(label, "black")

    marker_size = marker_size if marker_size is not None else SIZES["tree_marker"]
    marker_func = lambda x: (
        (marker_size, get_label_color(x.name)) if x.name is not None else None
    )

    ax.axes.get_yaxis().set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True, prune=None))
    ax.xaxis.set_tick_params(labelsize=SIZES["xlabel_tick"])
    ax.xaxis.label.set_size(SIZES["xlabel_font"])
    ax.set_title(
        title,
        x=0.01,
        y=1.0,
        ha="left",
        va="bottom",
        fontweight="bold",
        fontsize=16,
        zorder=10,
    )
    x_posns = _get_x_positions(input_tree)
    y_posns = _get_y_positions(
        input_tree, adjust=not hide_internal_nodes, outgroup=outgroup
    )
    xmax = max(x_posns.values())
    ax.set_xlim(-0.05 * xmax, 1.05 * xmax)
    top_margin = 0.5
    ymax = max(y_posns.values()) + top_margin
    ax.set_ylim(ymax, -0.5)
    ax_scale = ax.get_xlim()[1] - ax.get_xlim()[0]

    def value_to_str(value):
        if value is None or value == 0:
            return None
        return str(int(value)) if int(value) == value else str(value)

    if not branch_labels:
        if show_branch_lengths:

            def format_branch_label(x):
                return (
                    value_to_str(np.round(x.branch_length, 1))
                    if x.name != "root" and x.name is not None
                    else None
                )

        else:

            def format_branch_label(clade):
                return None

    elif isinstance(branch_labels, dict):

        def format_branch_label(clade):
            return branch_labels.get(clade)

    else:
        if not callable(branch_labels):
            raise TypeError(
                "branch_labels must be either a dict or a callable (function)"
            )

        def format_branch_label(clade):
            return value_to_str(branch_labels(clade))

    if show_branch_support:

        def format_support_value(clade):
            if clade.name == "root" or clade.name is None:
                return None
            try:
                confidences = clade.confidences
            except AttributeError:
                pass
            else:
                return "/".join(value_to_str(cnf.value) for cnf in confidences)
            return (
                value_to_str(clade.confidence) if clade.confidence is not None else None
            )

    def draw_clade_lines(
        use_linecollection: bool = False,
        orientation: str = "horizontal",
        y_here: float = 0,
        x_start: float = 0,
        x_here: float = 0,
        y_bot: float = 0,
        y_top: float = 0,
        color: str = "black",
        lw: float = 0.1,
    ) -> None:
        if use_linecollection and orientation == "horizontal":
            horizontal_lines.append([(x_start, y_here), (x_here, y_here)])
            horizontal_colors.append(color)
            horizontal_lws.append(lw)
        elif use_linecollection and orientation == "vertical":
            vertical_lines.append([(x_here, y_bot), (x_here, y_top)])
            vertical_colors.append(color)
            vertical_lws.append(lw)

    def draw_clade(clade: Any, x_start: float, default_color: str, lw: float) -> None:
        x_here = x_posns[clade]
        y_here = y_posns[clade]

        line_color = (
            clade_color_map.get(clade, default_color)
            if clade_color_map
            else default_color
        )

        if hasattr(clade, "color") and clade.color is not None:
            line_color = clade.color.to_hex()
        if hasattr(clade, "width") and clade.width is not None:
            lw = clade.width * plt.rcParams["lines.linewidth"]

        draw_clade_lines(
            use_linecollection=True,
            orientation="horizontal",
            y_here=y_here,
            x_start=x_start,
            x_here=x_here,
            color=line_color,
            lw=lw,
        )

        if marker_func is not None:
            marker = marker_func(clade)
            if (
                marker is not None
                and clade is not None
                and not (hide_internal_nodes and not clade.is_terminal())
            ):
                m_size, m_color = marker
                marker_x.append(x_here)
                marker_y.append(y_here)
                marker_sizes.append(m_size)
                marker_colors.append(m_color)

        label = label_func(str(clade.name))
        if label not in (None, clade.__class__.__name__) and not (
            hide_internal_nodes and not clade.is_terminal()
        ):
            text_x.append(x_here + min(0.02 * ax_scale, 1))
            text_y.append(y_here)
            texts.append(f" {label}")
            text_colors.append(get_label_color(label))

        if clade.clades:
            y_top_local = y_posns[clade.clades[0]]
            y_bot_local = y_posns[clade.clades[-1]]

            child_colors = []
            for child in clade.clades:
                child_line_color = (
                    clade_color_map.get(child, line_color)
                    if clade_color_map
                    else line_color
                )
                child_colors.append(child_line_color)

            if len(set(child_colors)) == 1:
                vertical_line_color = child_colors[0]
            else:
                vertical_line_color = line_color

            draw_clade_lines(
                use_linecollection=True,
                orientation="vertical",
                x_here=x_here,
                y_bot=y_bot_local,
                y_top=y_top_local,
                color=vertical_line_color,
                lw=lw,
            )

            for child in clade:
                draw_clade(child, x_here, line_color, lw)

    line_width = (
        line_width if line_width is not None else plt.rcParams["lines.linewidth"]
    )
    draw_clade(input_tree.root, 0, "k", line_width)

    if horizontal_lines:
        h_linecollection = mpcollections.LineCollection(
            horizontal_lines,
            colors=horizontal_colors,
            linewidths=horizontal_lws,
        )
        ax.add_collection(h_linecollection)

    if vertical_lines:
        v_linecollection = mpcollections.LineCollection(
            vertical_lines,
            colors=vertical_colors,
            linewidths=vertical_lws,
        )
        ax.add_collection(v_linecollection)

    if marker_x:
        ax.scatter(marker_x, marker_y, s=marker_sizes, c=marker_colors, zorder=3)

    for x, y, text, color in zip(text_x, text_y, texts, text_colors):
        ax.text(x, y, text, verticalalignment="center", color=color)

    ax.set_xlabel("branch length")
    ax.set_ylabel("taxa")

    for key, value in kwargs.items():
        try:
            list(value)
        except TypeError:
            raise ValueError(
                f'Keyword argument "{key}={value}" is not in the correct format.'
            ) from None

        if isinstance(value, dict):
            getattr(plt, str(key))(**dict(value))
        elif not (isinstance(value[0], tuple)):
            getattr(plt, str(key))(*value)
        elif isinstance(value[0], tuple):
            getattr(plt, str(key))(*value[0], **dict(value[1]))

    # --- New Code: Highlight cluster MRCA nodes and add a label box ---
    if cluster_mrcas and cluster_labels and len(cluster_mrcas) == len(cluster_labels):
        for mrca, cl_label in zip(cluster_mrcas, cluster_labels):
            # Get MRCA coordinates
            mx = x_posns[mrca]
            my = y_posns[mrca]

            # Draw a semi-transparent marker at the MRCA node
            # E.g., a circle marker with alpha=0.5 in the cluster's color
            node_color = clade_color_map.get(mrca, "red")
            ax.scatter(
                mx, my, s=100, c=node_color, alpha=0.5, zorder=5, edgecolors="none"
            )

            # Create a small box for the label to the right of the node
            box_width = 1.0  # Arbitrary width
            box_height = 0.5  # Arbitrary height
            box_x = mx + 0.1  # small offset from the node
            box_y = my - box_height / 2  # center vertically around the node

            # Add rectangle patch
            rect = Rectangle(
                (box_x, box_y),
                box_width,
                box_height,
                facecolor=node_color,
                alpha=0.5,
                zorder=4,
            )
            ax.add_patch(rect)

            # Add text inside the box
            ax.text(
                box_x + box_width / 2,
                box_y + box_height / 2,
                cl_label,
                ha="center",
                va="center",
                color="black",
                fontsize=8,
            )

    if output_name is not None:
        plt.savefig(output_name + ".png", bbox_inches="tight")

    return plt.gcf()

In [ ]:
from Bio import Phylo
import matplotlib.pyplot as plt
import numpy as np

# Define your clusters
clusters = [
    ["A", "B"],  # Cluster 1
    ["D", "C"],  # Cluster 2
    ["E", "F", "G", "H"],  # Cluster 3
]

# Assign colors for each cluster
cluster_colors = ["red", "blue", "green"]

# Create a map from clade to color
clade_color_map = {}
for i, cluster in enumerate(clusters):
    color = cluster_colors[i]
    mrca = tree.common_ancestor(cluster)

    # Color the entire subtree under this MRCA
    for descendant in mrca.find_clades():
        clade_color_map[descendant] = color
    # Remove the MRCA itself so its horizontal line doesn't take the cluster color
    if mrca in clade_color_map:
        del clade_color_map[mrca]

# Ensure you have the plot_tree function code defined above this point
plot_tree(
    tree,
    clade_color_map=clade_color_map,
    line_width=3,
    height_scale=2,
    width_scale=0.5,
    marker_size=0.005,
    hide_internal_nodes=True,
    show_terminal_labels=False,
    label_func=lambda x: "",
)

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib as mpl
import numpy as np
from typing import Any, Callable, Dict, List, Optional, Union

# Assuming plot_tree is already defined as per your provided code

# Sample Tree Structure
# For demonstration, we'll use Bio.Phylo's Tree object.
# You can replace this with your own tree structure compatible with plot_tree.

from Bio import Phylo
from io import StringIO

# Example Newick string
newick_str = "((A:0.1,B:0.2):0.3,(C:0.4,D:0.5):0.6);"

# Parse the Newick string using Bio.Phylo (ensure Biopython is installed)
tree = Phylo.read(StringIO(newick_str), "newick")

# Define a clade_color_map that maps all clades to 'black'
clade_color_map = {clade: "black" for clade in tree.find_clades()}


# Define label_colors as a callable that returns 'black' for any label
def black_label_color(label):
    return "black"


# Alternatively, using a lambda function:
# label_colors = lambda label: 'black'

# Create a figure with two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Plot the first tree normally on the first axis
plot_tree(
    input_tree=tree,
    clade_color_map=clade_color_map,
    label_colors=black_label_color,
    ax=axes[0],
    title="Original Tree",
    show_terminal_labels=True,
    show_branch_lengths=True,
    line_width=1.0,  # Optional: adjust line width
    marker_size=50,  # Optional: adjust marker size
    **{}
)

# Plot the second tree on the second axis
plot_tree(
    input_tree=tree,
    clade_color_map=clade_color_map,
    label_colors=black_label_color,
    ax=axes[1],
    title="Flipped Tree",
    show_terminal_labels=True,
    show_branch_lengths=True,
    line_width=1.0,  # Optional: adjust line width
    marker_size=50,  # Optional: adjust marker size
    **{}
)

# Invert the x-axis on the second plot to flip the tree horizontally
axes[1].invert_xaxis()

# Optional: Adjust layout for better spacing
plt.tight_layout()

# Display the plots
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mpl
import numpy as np
from matplotlib.patches import Polygon
from typing import Any, Callable, Dict, List, Optional, Union
from Bio import Phylo

COLORS = {
    "marker_terminal": "blue",
    "marker_internal": "darkgreen",
    "marker_normal": "red",
}
SIZES = {
    "tree_marker": 20,
    "xlabel_tick": 10,
    "xlabel_font": 12,
}
LINEWIDTHS = {
    "segment_boundary": 1.0,
}


def _get_x_positions(tree):
    x_dict = {}

    def assign_x(clade, current_x):
        x_dict[clade] = current_x
        for child in clade.clades:
            length = child.branch_length or 0.0
            assign_x(child, current_x + length)

    assign_x(tree.root, 0.0)
    return x_dict


def _get_y_positions(tree):
    """
    Returns a dict mapping each clade to its y-position in a simple top-to-bottom ordering
    of leaves. Internal nodes are placed at the midpoint of children.
    """
    y_dict = {}
    y_counter = 0

    def assign_y(clade):
        nonlocal y_counter
        for child in clade.clades:
            assign_y(child)
        if not clade.clades:
            # Leaf
            y_dict[clade] = y_counter
            y_counter += 1
        else:
            # internal node
            child_ys = [y_dict[ch] for ch in clade.clades]
            y_dict[clade] = np.mean(child_ys)

    assign_y(tree.root)
    return y_dict


def plot_tree(
    input_tree: Any,
    ax: Optional[plt.Axes] = None,
    title: str = "",
    show_terminal_labels: bool = True,
    line_width: Optional[float] = None,
    marker_size: Optional[int] = None,
    # Triangle tiling parameters
    tile_leaf_triangles: bool = True,
    triangles_on: str = "right",  # "left" or "right"
    triangle_offset: float = 2.0,
    triangle_color: str = "red",
    triangle_alpha: float = 0.5,
    # Let user supply a list of leaf labels (clusters)
    leaf_labels: Optional[List[str]] = None,
    **kwargs,
) -> plt.Figure:
    """
    Plot the tree. Optionally tile 'fat' triangles from leaf to leaf, so consecutive leaf triangles
    meet edge to edge (no leftover space at top/bottom). Triangles can be on "right" or "left" side.
    Additionally, optionally label each leaf-triangle with provided cluster labels.

    :param input_tree: Bio.Phylo tree or similar
    :param ax: optional matplotlib axis
    :param title: figure title
    :param show_terminal_labels: draw leaf node labels
    :param line_width: thickness of drawn lines
    :param marker_size: size of node markers
    :param tile_leaf_triangles: if True, create side-by-side tiling of triangles
    :param triangles_on: "right" (default) or "left" for horizontal alignment
    :param triangle_offset: distance from furthest leaf (if right) or from leftmost leaf (if left)
    :param triangle_color: color of triangles
    :param triangle_alpha: alpha transparency
    :param leaf_labels: optional list of labels to place on triangles. e.g. ["Cluster 1", "Cluster 2", ...]
                        If fewer than leaves, we cycle them or fill the rest automatically.
    """
    import matplotlib.collections as mpcollections

    # set defaults
    line_width = line_width or LINEWIDTHS["segment_boundary"]
    marker_size = marker_size or SIZES["tree_marker"]

    # get x, y
    x_posns = _get_x_positions(input_tree)
    y_posns = _get_y_positions(input_tree)

    leaves = input_tree.get_terminals()
    max_x = max(x_posns[lf] for lf in leaves)
    min_x = min(x_posns[lf] for lf in leaves)

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    else:
        fig = ax.figure

    # basic axis setup
    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.axes.get_yaxis().set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)

    x_range = max_x - min_x
    y_max = max(y_posns.values())
    ax.set_xlim(min_x - 0.05 * x_range, max_x + 1.05 * x_range)
    ax.set_ylim(y_max + 0.5, -0.5)

    horizontal_lines = []
    vertical_lines = []
    horizontal_colors = []
    vertical_colors = []
    horizontal_lws = []
    vertical_lws = []

    marker_x = []
    marker_y = []
    marker_sizes = []
    marker_colors = []

    text_x = []
    text_y = []
    texts = []
    text_colors = []

    def draw_line(orientation, xstart, xend, yval, ybot, ytop, color, lw):
        if orientation == "horizontal":
            horizontal_lines.append([(xstart, yval), (xend, yval)])
            horizontal_colors.append(color)
            horizontal_lws.append(lw)
        else:
            vertical_lines.append([(xend, ybot), (xend, ytop)])
            vertical_colors.append(color)
            vertical_lws.append(lw)

    def draw_clade(clade, xstart, color, lw):
        x_here = x_posns[clade]
        y_here = y_posns[clade]

        # horizontal
        draw_line("horizontal", xstart, x_here, y_here, 0, 0, color, lw)

        if not clade.clades:
            # leaf
            marker_x.append(x_here)
            marker_y.append(y_here)
            marker_sizes.append(marker_size)
            marker_colors.append(color)
            if show_terminal_labels and clade.name:
                text_x.append(x_here + 0.02 * x_range)
                text_y.append(y_here)
                texts.append(" " + clade.name)
                text_colors.append("black")
        else:
            # internal
            marker_x.append(x_here)
            marker_y.append(y_here)
            marker_sizes.append(marker_size * 0.6)
            marker_colors.append(color)

        if clade.clades:
            ytop = y_posns[clade.clades[0]]
            ybot = y_posns[clade.clades[-1]]
            draw_line("vertical", 0, x_here, 0, ybot, ytop, color, lw)
            for child in clade.clades:
                draw_clade(child, x_here, color, lw)

    # draw from root
    def draw_root():
        draw_clade(input_tree.root, 0.0, "black", line_width)

    draw_root()

    # add linecollections
    if horizontal_lines:
        hc = mpcollections.LineCollection(
            horizontal_lines, colors=horizontal_colors, linewidths=horizontal_lws
        )
        ax.add_collection(hc)
    if vertical_lines:
        vc = mpcollections.LineCollection(
            vertical_lines, colors=vertical_colors, linewidths=vertical_lws
        )
        ax.add_collection(vc)

    # scatter markers
    if marker_x:
        ax.scatter(marker_x, marker_y, s=marker_sizes, c=marker_colors, zorder=3)

    # add text
    for xx, yy, txt, c in zip(text_x, text_y, texts, text_colors):
        ax.text(xx, yy, txt, color=c, va="center")

    # If user wants side-by-side "fat" triangles that fill the entire top/bottom with no space
    if tile_leaf_triangles and len(leaves) > 0:
        # sort leaves by y
        leaves_sorted = sorted(leaves, key=lambda lf: y_posns[lf])
        # We'll define top=the midpoint between leaf i and i-1,
        # but for i=0, top is the same as the leaf's y => no space above
        # Similarly for bottom at midpoint with i+1

        # figure out final label list
        if leaf_labels is None:
            # default to "Cluster 1", "Cluster 2", ...
            leaf_labels = [f"Cluster {i+1}" for i in range(len(leaves_sorted))]
        if len(leaf_labels) < len(leaves_sorted):
            # either cycle or just repeat last
            # For simplicity, let's just repeat last
            needed = len(leaves_sorted) - len(leaf_labels)
            leaf_labels = leaf_labels + [leaf_labels[-1]] * needed

        # define the common boundary
        if triangles_on.lower() == "right":
            x_tri_line = max_x + triangle_offset
        else:
            # left
            x_tri_line = min_x - triangle_offset

        # We'll add labels inside each triangle. We'll define a small offset inside
        label_x_offset = (
            0.1  # how far horizontally from the right boundary we place text
        )
        label_y_offset = 0.0

        for i, leaf in enumerate(leaves_sorted):
            lx = x_posns[leaf]
            ly = y_posns[leaf]

            # figure top
            if i == 0:
                top = ly
            else:
                prev_leaf = leaves_sorted[i - 1]
                top = 0.5 * (ly + y_posns[prev_leaf])
            # figure bottom
            if i == len(leaves_sorted) - 1:
                bottom = ly
            else:
                next_leaf = leaves_sorted[i + 1]
                bottom = 0.5 * (ly + y_posns[next_leaf])

            # Ensure top < bottom
            upper = min(top, bottom)
            lower = max(top, bottom)
            # For no leftover space, the top leaf (i=0) uses top=the leaf's y, and the bottom leaf (i=last) uses bottom=the leaf's y.

            if triangles_on.lower() == "right":
                # 3 vertices: (lx, ly), (x_tri_line, upper), (x_tri_line, lower)
                tri_vertices = [
                    (lx, ly),
                    (x_tri_line, upper),
                    (x_tri_line, lower),
                ]
            else:
                # left
                # 3 vertices: (lx, ly), (x_tri_line, upper), (x_tri_line, lower) but the shape is reversed horizontally
                tri_vertices = [
                    (lx, ly),
                    (x_tri_line, upper),
                    (x_tri_line, lower),
                ]

            tri_poly = Polygon(
                tri_vertices,
                closed=True,
                facecolor=triangle_color,
                alpha=triangle_alpha,
                edgecolor="none",
                zorder=5,
            )
            ax.add_patch(tri_poly)

            # place label inside the triangle
            label_str = leaf_labels[i]
            # We'll place the text near the "right" side or "left" side depending on triangles_on
            if triangles_on.lower() == "right":
                # near x_tri_line - label_x_offset
                textx = x_tri_line - label_x_offset
            else:
                textx = x_tri_line + label_x_offset

            texty = 0.5 * (upper + lower) + label_y_offset

            ax.text(
                textx,
                texty,
                label_str,
                ha="right" if triangles_on.lower() == "right" else "left",
                va="center",
                color="black",
                fontsize=9,
                zorder=10,
            )

    return ax.figure

from io import StringIO
from Bio import Phylo
import matplotlib.pyplot as plt

# Construct sample tree
newick_str = "((A:0.1,B:0.2):0.3,(C:0.4,D:0.5):0.6);"
tree = Phylo.read(StringIO(newick_str), "newick")

# We define some cluster labels for the leaves
leaves = tree.get_terminals()
leaf_names = [lf.name for lf in leaves]  # e.g. A,B,C,D
my_labels = [f"Cluster {i+1}" for i in range(len(leaves))]  # custom

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Normal orientation on left
plot_tree(
    input_tree=tree,
    ax=axes[0],
    title="Side-by-side Triangles (Right)",
    tile_leaf_triangles=True,
    triangles_on="right",
    triangle_offset=1.0,
    triangle_color="green",
    leaf_labels=my_labels,  # label each leaf's tile
)

# Flipped orientation on right
plot_tree(
    input_tree=tree,
    ax=axes[1],
    title="Side-by-side Triangles (Left) [Flipped axis]",
    tile_leaf_triangles=True,
    triangles_on="left",  # place triangles to the left of min_x
    triangle_offset=1.0,
    triangle_color="orange",
    leaf_labels=my_labels,
)
axes[1].invert_xaxis()  # so it looks away from the tree

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from Bio import Phylo
from matplotlib.patches import Polygon, Rectangle
from io import StringIO
from typing import List, Optional

########################
# 1) Tree utilities
########################


def get_x_positions(tree):
    """
    Return a dict mapping each clade to its x-position (distance from root).
    We'll forcibly set the root's branch_length=0 to avoid root stubs.
    """
    # ensure root length is zero
    if tree.root.branch_length:
        tree.root.branch_length = 0.0

    x_dict = {}

    def assign_x(clade, curx):
        x_dict[clade] = curx
        for child in clade.clades:
            bl = child.branch_length or 0.0
            assign_x(child, curx + bl)

    assign_x(tree.root, 0.0)
    return x_dict


def get_y_positions(tree):
    """
    Return a dict mapping each clade to a y-position in top-to-bottom ordering.
    Leaves are assigned consecutive integers, internal nodes at the mean of children.
    """
    y_dict = {}
    y_counter = 0

    def assign_y(clade):
        nonlocal y_counter
        for child in clade.clades:
            assign_y(child)
        if not clade.clades:
            # leaf
            y_dict[clade] = y_counter
            y_counter += 1
        else:
            kids = [y_dict[ch] for ch in clade.clades]
            y_dict[clade] = np.mean(kids)

    assign_y(tree.root)
    return y_dict


def flip_x_positions(x_dict):
    """
    Reflect x-positions around the midpoint so the tree is reversed horizontally.
    """
    xs = list(x_dict.values())
    xmin, xmax = min(xs), max(xs)
    mid = 0.5 * (xmin + xmax)
    for c in x_dict:
        x_orig = x_dict[c]
        x_dict[c] = mid - (x_orig - mid)


########################
# 2) Plot the tree lines (no leaf markers/names)
########################


def plot_tree(
    tree,
    ax: plt.Axes,
    flip_x: bool = False,
    line_color: str = "black",
    line_width: float = 1.0,
):
    """
    Draw a simple horizontal cladogram in 'ax', with no leaf markers or leaf names.
    If flip_x=True, reflect the x-positions around the midpoint for a reversed view.
    Return (x_dict, y_dict).
    """
    x_dict = get_x_positions(tree)
    y_dict = get_y_positions(tree)
    if flip_x:
        flip_x_positions(x_dict)

    # gather lines
    h_lines = []
    v_lines = []

    def add_hline(x1, x2, y):
        h_lines.append(((x1, y), (x2, y)))

    def add_vline(x, y1, y2):
        v_lines.append(((x, y1), (x, y2)))

    def draw_clade(clade, xstart):
        x_here = x_dict[clade]
        y_here = y_dict[clade]
        # horizontal
        add_hline(xstart, x_here, y_here)
        if clade.clades:
            y_top = y_dict[clade.clades[0]]
            y_bot = y_dict[clade.clades[-1]]
            add_vline(x_here, y_bot, y_top)
            for child in clade.clades:
                draw_clade(child, x_here)

    draw_clade(tree.root, 0.0)

    import matplotlib.collections as mc

    hc = mc.LineCollection(h_lines, colors=line_color, linewidths=line_width)
    vc = mc.LineCollection(v_lines, colors=line_color, linewidths=line_width)
    ax.add_collection(hc)
    ax.add_collection(vc)

    # define x/y lims
    all_x = list(x_dict.values())
    xmin, xmax = min(all_x), max(all_x)
    all_y = list(y_dict.values())
    ymin, ymax = min(all_y), max(all_y)

    xmargin = 0.1 * (xmax - xmin) if (xmax > xmin) else 1
    ymargin = 0.5
    ax.set_xlim(xmin - xmargin, xmax + xmargin)
    ax.set_ylim(ymax + ymargin, ymin - ymargin)

    # remove spines/ticks
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

    return x_dict, y_dict


########################
# 3) Uniform Triangles with Boxed Labels
########################


def add_uniform_triangles(
    ax: plt.Axes,
    x_dict: dict,
    y_dict: dict,
    leaves: list,
    side: str = "right",
    triangle_offset: float = 1.0,
    triangle_color: str = "red",
    triangle_alpha: float = 0.5,
    cluster_labels: Optional[List[str]] = None,
    label_fontsize: int = 10,
):
    """
    For each leaf, add a uniform-height triangle that fills the top->bottom range with no leftover space.
    Then place a semi-transparent box (alpha=0.3) with the cluster label inside.
    """
    if not leaves:
        return
    N = len(leaves)
    leaves_sorted = sorted(leaves, key=lambda lf: y_dict[lf])
    yvals = [y_dict[lf] for lf in leaves_sorted]
    y_min, y_max = yvals[0], yvals[-1]

    global_pad = 0.1 * (y_max - y_min)
    chunk = ((y_max - y_min) + 2 * global_pad) / N

    if not cluster_labels:
        cluster_labels = [f"Cluster {i+1}" for i in range(N)]
    if len(cluster_labels) < N:
        cluster_labels += [cluster_labels[-1]] * (N - len(cluster_labels))

    # define line x for the triangles
    if side == "right":
        x_leaf_max = max(x_dict[lf] for lf in leaves)
        tri_x = x_leaf_max + triangle_offset
        # We'll place the label box to the left of tri_x a bit
        label_box_dx = -0.01
    else:
        x_leaf_min = min(x_dict[lf] for lf in leaves)
        tri_x = x_leaf_min - triangle_offset
        label_box_dx = 0.01

    # define approximate box width as fraction of the subplot's x-range
    # We'll measure axis data-lims:
    xlim = ax.get_xlim()
    data_width = xlim[1] - xlim[0]
    box_w_fraction = 0.05  # 5% of axis
    box_w = box_w_fraction * data_width

    for i, leaf in enumerate(leaves_sorted):
        top = (y_min - global_pad) + i * chunk
        bottom = top + chunk

        lx = x_dict[leaf]
        ly = y_dict[leaf]

        if side == "right":
            tri_vertices = [(lx, ly), (tri_x, top), (tri_x, bottom)]
        else:
            tri_vertices = [(lx, ly), (tri_x, top), (tri_x, bottom)]

        from matplotlib.patches import Polygon, Rectangle

        tri_poly = Polygon(
            tri_vertices,
            closed=True,
            facecolor=triangle_color,
            alpha=triangle_alpha,
            edgecolor="none",
            zorder=5,
        )
        ax.add_patch(tri_poly)

        # label box
        midy = 0.5 * (top + bottom)
        # Let's define the box height as 0.6 of chunk
        box_h = 0.6 * chunk

        # We'll place the box so that its center is midy
        box_y = midy - 0.5 * box_h
        if side == "right":
            # box at tri_x + label_box_dx - box_w
            # so that it doesn't overlap the triangle too much
            box_x = tri_x + label_box_dx - box_w
        else:
            # side=="left"
            # box at tri_x + label_box_dx
            box_x = tri_x + label_box_dx

        rect = Rectangle(
            (box_x, box_y),
            box_w,
            box_h,
            facecolor=triangle_color,
            alpha=0.3,
            edgecolor="none",
            zorder=6,
        )
        ax.add_patch(rect)

        # text in middle
        lbl = cluster_labels[i]
        ax.text(
            box_x + 0.5 * box_w,
            box_y + 0.5 * box_h,
            lbl,
            ha="center",
            va="center",
            fontsize=label_fontsize,
            color="black",
            zorder=7,
        )


########################
# 4) Single Scale Bar for the Figure
########################


def pick_scale_bar_length(xrange: float) -> float:
    """
    Given an x-range, pick a 'nice' round scale bar that is ~ 20% of that range.
    e.g. 0.1, 1, 10, etc.
    """
    desired = 0.2 * xrange
    # pick a power-of-10 style
    # We'll find 10^n <= desired < 10^(n+1) etc.
    # Then we possibly do 2 or 5 times that. Or keep it simpler
    # We'll do a quick approach:
    p = 10 ** np.floor(np.log10(desired))
    # next factor
    m = desired / p
    if m < 2:
        bar_len = p
    elif m < 5:
        bar_len = 2 * p
    else:
        bar_len = 5 * p
    return bar_len


def add_single_scale_bar(
    ax: plt.Axes,
    all_trees,  # list of trees to consider
    location="bottomleft",
    line_width=2.0,
    color="black",
    fontsize=10,
):
    """
    Compute the union x-lims of all_trees (in data coords) to pick a scale bar length,
    then place a single bar in 'ax' at the corner with that length.
    We'll assume all_trees were drawn in the same data coords, or that we only want to
    scale the first tree's coords. Typically, for 2 subplots, we do it in the left subplot.
    """
    # We'll gather min_x, max_x across all trees
    min_x = float("inf")
    max_x = float("-inf")
    for d in all_trees:
        # d is e.g. x_dict from plot_tree
        xvals = list(d.values())
        mi, ma = min(xvals), max(xvals)
        if mi < min_x:
            min_x = mi
        if ma > max_x:
            max_x = ma
    # define the bar length
    data_range = max_x - min_x
    bar_len = pick_scale_bar_length(data_range)
    # We'll place it in the specified location
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x_range = xlim[1] - xlim[0]
    y_range = ylim[0] - ylim[1] if ylim[0] > ylim[1] else ylim[1] - ylim[0]
    xmarg = 0.02 * x_range
    ymarg = 0.02 * y_range

    if "bottom" in location:
        # check if yaxis is inverted
        if ax.yaxis_inverted():
            y0 = ylim[0] + ymarg
        else:
            y0 = ylim[1] - ymarg
    else:
        # top
        if ax.yaxis_inverted():
            y0 = ylim[1] - ymarg
        else:
            y0 = ylim[0] + ymarg

    if "left" in location:
        x0 = xlim[0] + xmarg
    else:
        x0 = xlim[1] - xmarg - bar_len

    x1 = x0 + bar_len
    # draw line
    ax.plot([x0, x1], [y0, y0], lw=line_width, color=color, zorder=10)
    # label
    label_str = f"{bar_len:g}"
    ax.text(
        0.5 * (x0 + x1),
        y0,
        label_str,
        ha="center",
        va="bottom" if not ax.yaxis_inverted() else "top",
        fontsize=fontsize,
        color=color,
        zorder=11,
    )


########################
# 5) Putting it all Together
########################


def main():
    newick_str = "((A:0.1,B:0.2):0.3,(C:0.4,D:0.5):0.6);"
    tree_left = Phylo.read(StringIO(newick_str), "newick")
    tree_right = Phylo.read(StringIO(newick_str), "newick")

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # Left subplot: normal orientation, triangles on right
    x_dict_left, y_dict_left = plot_tree(tree_left, ax=axes[0], flip_x=False)
    leaves_left = tree_left.get_terminals()
    add_uniform_triangles(
        ax=axes[0],
        x_dict=x_dict_left,
        y_dict=y_dict_left,
        leaves=leaves_left,
        side="right",
        triangle_offset=0.3,
        triangle_color="green",
        cluster_labels=["C1", "C2", "C3", "C4"],
    )
    axes[0].set_title("Left Subplot: Normal Tree\nTriangles on Right", fontsize=11)

    # Right subplot: flip data horizontally, triangles on left
    x_dict_right, y_dict_right = plot_tree(tree_right, ax=axes[1], flip_x=True)
    leaves_right = tree_right.get_terminals()
    add_uniform_triangles(
        ax=axes[1],
        x_dict=x_dict_right,
        y_dict=y_dict_right,
        leaves=leaves_right,
        side="left",
        triangle_offset=0.3,
        triangle_color="orange",
        cluster_labels=["C1", "C2", "C3", "C4"],
    )
    axes[1].set_title("Right Subplot: Flipped Tree\nTriangles on Left", fontsize=11)

    # Single scale bar in the left subplot
    # We pass a list of x_dict for both trees so we get the full range
    all_xdicts = [x_dict_left, x_dict_right]
    add_single_scale_bar(
        ax=axes[0],
        all_trees=all_xdicts,
        location="bottomleft",
        line_width=2.0,
        color="black",
        fontsize=9,
    )

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
newick_str = "((A:0.1,B:0.2):0.3,(C:0.4,D:0.5):0.6);"
tree_left = Phylo.read(StringIO(newick_str), "newick")
tree_right = Phylo.read(StringIO(newick_str), "newick")
Phylo.draw(tree_left)

In [ ]:
def add_uniform_triangles(
    ax: plt.Axes,
    x_dict: dict,
    y_dict: dict,
    all_clades: list,
    side: str = "right",
    triangle_offset: float = 1.0,
    triangle_color: str = "red",
    triangle_alpha: float = 0.5,
    cluster_labels: Optional[List[str]] = None,
    label_fontsize: int = 10,
    # NEW parameters:
    triangle_y_ratio: float = 1.0,
    label_box_aspect: float = 2.0,
    label_box_offset: float = 0.02,
):
    """
    Draw a set of triangles for each leaf, all with the SAME vertical 'chunk' size,
    and place the leaf's y position at the CENTER of that chunk (=> symmetrical).

    Additionally:
    - 'triangle_y_ratio' multiplies the normal chunk height, letting you
      make triangles taller or shorter relative to spacing among leaves.
    - The cluster label box is placed OUTSIDE the triangle edge with an aspect
      ratio 'label_box_aspect' => width = aspect * height. We also offset
      horizontally by 'label_box_offset' in data coords so it doesn't overlap.

    'side': "right" or "left" => whether triangles extend outward from the leaf x.

    Steps:
      1) Identify LEAVES only => top leaf to bottom leaf => uniform chunk among them.
      2) For each leaf i, define a 'middle_y = y_min + (i+0.5)*chunk' so leaf apex is center.
      3) The actual triangle top/bottom = middle_y ± (chunk * triangle_y_ratio / 2).
      4) The apex is at (leaf_x, leaf_y) for the leaf, and the far side is (tri_x, top) + (tri_x, bottom).
      5) The label box is placed outside the triangle, with a chosen aspect ratio (width > height).
    """

    # 1) Gather leaves
    leaves = [c for c in all_clades if not c.clades]
    if not leaves:
        return
    # Sort leaves by ascending y
    leaves_sorted = sorted(leaves, key=lambda lf: y_dict[lf])
    N = len(leaves_sorted)

    # 2) Get top & bottom leaf y, ignoring internal nodes => uniform chunk among leaves
    y_min_leaf = y_dict[leaves_sorted[0]]
    y_max_leaf = y_dict[leaves_sorted[-1]]

    # 3) Optionally define a global pad
    global_pad = 0.1*(y_max_leaf - y_min_leaf)
    y_min = y_min_leaf - global_pad
    y_max = y_max_leaf + global_pad
    total_leaf_range = (y_max - y_min)

    # 4) 'chunk' is the base vertical slice, ignoring 'triangle_y_ratio' for now
    base_chunk = total_leaf_range / N

    # 5) If user doesn't specify cluster_labels, create defaults
    if not cluster_labels:
        cluster_labels = [f"Cluster {i+1}" for i in range(N)]
    if len(cluster_labels) < N:
        cluster_labels += [cluster_labels[-1]]*(N - len(cluster_labels))

    # 6) Compute tri_x line depending on side
    if side == "right":
        x_leaf_max = max(x_dict[l] for l in leaves)
        tri_x = x_leaf_max + triangle_offset
        # Label box is placed further right
        # We shift by + label_box_offset from tri_x
    else:
        x_leaf_min = min(x_dict[l] for l in leaves)
        tri_x = x_leaf_min - triangle_offset
        # We'll place labels further left by label_box_offset

    # 7) We'll measure axis data-lims to help define the label box size
    xlim = ax.get_xlim()
    data_w = xlim[1] - xlim[0]

    from matplotlib.patches import Polygon, Rectangle

    # 8) Build the triangles in ascending leaf order
    for i, leaf in enumerate(leaves_sorted):
        leaf_x = x_dict[leaf]
        leaf_y = y_dict[leaf]

        # Middle of chunk 'i'
        # The apex's vertical chunk is:
        #   mid_y = y_min + (i+0.5)*base_chunk
        # Then we multiply by 'triangle_y_ratio' to get the final height
        mid_y = y_min + (i + 0.5) * base_chunk
        half_height = (base_chunk * triangle_y_ratio) / 2.0

        tri_top = mid_y - half_height
        tri_bottom = mid_y + half_height

        # The triangle has apex at (leaf_x, leaf_y)
        # Far side is (tri_x, tri_top) and (tri_x, tri_bottom)
        if side == "right":
            tri_verts = [(leaf_x, leaf_y), (tri_x, tri_top), (tri_x, tri_bottom)]
        else:
            tri_verts = [(leaf_x, leaf_y), (tri_x, tri_top), (tri_x, tri_bottom)]

        poly = Polygon(
            tri_verts,
            closed=True,
            facecolor=triangle_color,
            alpha=triangle_alpha,
            edgecolor="none",
            zorder=5
        )
        ax.add_patch(poly)

        # 9) Define a label box outside the triangle
        #    We'll make it 'box_height = 0.6 * (base_chunk * triangle_y_ratio)'
        #    then 'box_width = label_box_aspect * box_height'
        box_height = 0.6 * (base_chunk * triangle_y_ratio)
        box_width  = label_box_aspect * box_height

        # The label box is centered around mid_y
        box_y = mid_y - box_height/2.0

        # The horizontal placement depends on side
        if side == "right":
            # We'll place it offset further to the right from tri_x
            box_x = tri_x + label_box_offset
        else:
            # side=left => place it left from tri_x - box_width - label_box_offset
            box_x = tri_x - label_box_offset - box_width

        rect = Rectangle(
            (box_x, box_y),
            box_width, box_height,
            facecolor=triangle_color,
            alpha=0.3,
            edgecolor="none",
            zorder=6
        )
        ax.add_patch(rect)

        # 10) Place the text inside
        ax.text(
            box_x + 0.5*box_width,
            box_y + 0.5*box_height,
            cluster_labels[i],
            ha="center", va="center",
            fontsize=label_fontsize,
            color="black",
            zorder=7
        )

    # (Optional) re-adjust y-lims to ensure we see top & bottom
    ax.set_ylim(y_min, y_max)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from Bio import Phylo
from matplotlib.patches import Polygon, Rectangle
from io import StringIO
from typing import List, Optional

#################################
# 1) UTILS: get_x_positions, get_y_positions, flip_x_positions
#################################

def get_x_positions(tree):
    """
    Return a dict mapping each clade to its x-position (distance from root),
    forcing the root's branch_length=0 to avoid extra stubs.
    """
    # Force root branch length = 0
    if tree.root.branch_length:
        tree.root.branch_length = 0.0

    x_dict = {}
    def assign_x(clade, curx):
        x_dict[clade] = curx
        for child in clade.clades:
            bl = child.branch_length or 0.0
            assign_x(child, curx + bl)
    assign_x(tree.root, 0.0)
    return x_dict


def get_y_positions(tree):
    """
    Return a dict mapping each clade to a y-position in top-to-bottom order for leaves.
    Internal nodes go at the mean of their children.
    """
    y_dict = {}
    y_counter = 0
    def assign_y(clade):
        nonlocal y_counter
        for child in clade.clades:
            assign_y(child)
        if not clade.clades:  # leaf
            y_dict[clade] = y_counter
            y_counter += 1
        else:
            child_ys = [y_dict[ch] for ch in clade.clades]
            y_dict[clade] = np.mean(child_ys)
    assign_y(tree.root)
    return y_dict


def flip_x_positions(x_dict):
    """
    Reflect the x-values around their midpoint so the tree is horizontally reversed.
    """
    xs = list(x_dict.values())
    xmin, xmax = min(xs), max(xs)
    mid = 0.5*(xmin + xmax)
    for c in x_dict:
        old = x_dict[c]
        x_dict[c] = mid - (old - mid)

def scale_x_positions(x_dict, factor: float):
    """
    Multiply all x-coordinates by 'factor' to scale up small branch lengths visually.
    This does not change the 'units' logically, but we must handle the scale bar carefully.
    """
    for c in x_dict:
        x_dict[c] *= factor

#################################
# 2) PLOTTING THE TREE LINES ONLY
#################################

def plot_tree(
    tree,
    ax: plt.Axes,
    flip_x: bool=False,
    scale_factor: float=1.0,
    line_color: str="black",
    line_width: float=1.0
):
    """
    Draws a cladogram (horizontal lines + vertical lines for branches),
    no leaf markers or names. If flip_x=True, data-coords are mirrored horizontally.
    Also multiply all x positions by 'scale_factor'.
    Returns (x_dict, y_dict, scale_factor).
    """
    # 1) get original x,y
    x_dict = get_x_positions(tree)
    y_dict = get_y_positions(tree)

    # 2) optionally flip
    if flip_x:
        flip_x_positions(x_dict)
    # 3) optionally scale up
    if scale_factor != 1.0:
        scale_x_positions(x_dict, scale_factor)

    # gather line segments
    h_segments = []
    v_segments = []

    def add_hline(x1, x2, y):
        h_segments.append(((x1,y),(x2,y)))
    def add_vline(x, y1, y2):
        # skip near-zero lines
        if abs(y2 - y1) < 1e-9:
            return
        v_segments.append(((x,y1),(x,y2)))

    def draw_clade(clade, xstart, is_root=False):
        xh = x_dict[clade]
        yh = y_dict[clade]
        if not is_root:
            add_hline(xstart, xh, yh)
        if clade.clades:
            y_bot = y_dict[clade.clades[-1]]
            y_top = y_dict[clade.clades[0]]
            v_segments.append(((xh,y_bot),(xh,y_top)))
            for child in clade.clades:
                draw_clade(child, xh)

    draw_clade(tree.root, 0.0, is_root=True)

    import matplotlib.collections as mc
    hc = mc.LineCollection(h_segments, colors=line_color, linewidths=line_width)
    vc = mc.LineCollection(v_segments, colors=line_color, linewidths=line_width)
    ax.add_collection(hc)
    ax.add_collection(vc)

    # define lims
    all_x = list(x_dict.values())
    all_y = list(y_dict.values())
    xmin, xmax = min(all_x), max(all_x)
    ymin, ymax = min(all_y), max(all_y)

    xmargin = 0.1*(xmax - xmin) if (xmax>xmin) else 1
    ymargin = 0.5
    ax.set_xlim(xmin - xmargin, xmax + xmargin)
    ax.set_ylim(ymax + ymargin, ymin - ymargin)

    # remove ticks/spines
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

    return x_dict, y_dict, scale_factor

#################################
# 3) UNIFORM TRIANGLES + LABELS OUTSIDE
#################################

def add_uniform_triangles(
    ax: plt.Axes,
    x_dict: dict,
    y_dict: dict,
    all_clades: list,
    side: str = "right",
    triangle_offset: float=1.0,
    triangle_color: str="red",
    triangle_alpha: float=0.5,
    cluster_labels: Optional[List[str]]=None,
    label_fontsize: int=10
):
    """
    - Compute bounding box from leaves only => uniform chunk in vertical dimension.
    - Triangles each have the same vertical height.
    - Place label boxes *outside* the triangle (to the extreme right or left).

    The top leaf gets the first chunk, the bottom leaf gets the last chunk,
    so they exactly fill the leaf range with optional padding.
    """
    # gather leaves
    leaves = [c for c in all_clades if not c.clades]
    if not leaves:
        return
    leaves_sorted = sorted(leaves, key=lambda lf: y_dict[lf])
    N = len(leaves_sorted)
    # bounding box from topmost leaf to bottommost leaf
    y_min_leaf = y_dict[leaves_sorted[0]]
    y_max_leaf = y_dict[leaves_sorted[-1]]

    # small pad
    global_pad = 0.2*(y_max_leaf - y_min_leaf)
    y_min = y_min_leaf - global_pad
    y_max = y_max_leaf + global_pad

    total_height = y_max - y_min
    chunk = total_height / N

    # cluster labels
    if not cluster_labels:
        cluster_labels = [f"Cluster {i+1}" for i in range(N)]
    if len(cluster_labels)<N:
        cluster_labels += [cluster_labels[-1]]*(N-len(cluster_labels))

    # define tri_x line
    from math import inf
    if side=="right":
        x_leaf_max = max(x_dict[l] for l in leaves)
        tri_x = x_leaf_max + triangle_offset
        # place label box further out
        label_box_dx = +5 # shift label box to the *right* of tri_x
    else:
        x_leaf_min = min(x_dict[l] for l in leaves)
        tri_x = x_leaf_min - triangle_offset
        # shift label box slightly more to the left
        label_box_dx = -5

    # measure axis data-lims for label box size
    xlim = ax.get_xlim()
    data_w = xlim[1]-xlim[0]
    box_aspect = 2.0
    box_w_fraction = 0.08
    box_w = box_w_fraction*data_w

    from matplotlib.patches import Polygon, Rectangle
    for i, leaf in enumerate(leaves_sorted):
        top = y_min + i*chunk
        bottom = top + chunk

        lx = x_dict[leaf]
        ly = y_dict[leaf]

        # define triangle
        if side=="right":
            tri_verts = [(lx, ly),(tri_x, top),(tri_x, bottom)]
        else:
            tri_verts = [(lx, ly),(tri_x, top),(tri_x, bottom)]

        poly = Polygon(
            tri_verts,
            closed=True,
            facecolor=triangle_color,
            alpha=triangle_alpha,
            edgecolor="none",
            zorder=5
        )
        ax.add_patch(poly)

        # place label box entirely outside the triangle
        midy = 0.5*(top+bottom)
        box_h = 0.6*chunk
        box_y = midy - 0.5*box_h

        if side=="right":
            # label box to the right of tri_x
            box_x = tri_x + label_box_dx
        else:
            # label box to the left of tri_x minus box_w
            box_x = tri_x + 5 + label_box_dx - box_w

        rect = Rectangle(
            (box_x, box_y),
            box_w, box_h,
            facecolor=triangle_color,
            alpha=0.3,
            edgecolor="none",
            zorder=6
        )
        ax.add_patch(rect)

        ax.text(
            box_x + 0.5*box_w,
            box_y + 0.5*box_h,
            cluster_labels[i],
            ha="center", va="center",
            fontsize=label_fontsize, color="black",
            zorder=7
        )

    # optionally re-adjust y-lims so we see entire top/bottom
    ax.set_ylim(y_min, y_max)

#################################
# 4) SINGLE SCALE BAR WITH SCALING
#################################

def pick_scale_length(xrange: float) -> float:
    """
    Return a 'nice' scale bar ~20% of xrange, e.g. 0.1, 0.2, 0.5, 1, 2, 5, 10, ...
    """
    if xrange <= 0:
        return 0.1
    target = 0.2*xrange
    p = 10**(np.floor(np.log10(target)))
    factor = target/p
    if factor<2:
        return p
    elif factor<5:
        return 2*p
    else:
        return 5*p

def add_scale_bar(
    ax: plt.Axes,
    x_dicts: List[dict],
    scale_factors: List[float],
    location="bottomleft",
    line_width=2.0,
    color="black",
    fontsize=10,
):
    """
    Combine x-ranges from x_dicts in original scale. Then pick a scale bar ~20%.
    BUT we must multiply that bar length by the scale factor used for data coords.

    x_dicts: the x-dicts after scaling
    scale_factors: each x-dict's scale factor
    We assume all subplots used the same scale factor or we pick the first.

    By default, if you are using the same scale_factor for both trees, we can
    do scale_factor = scale_factors[0] and ignore the second.
    """
    if not scale_factors:
        return
    # assume the same factor for both if user wants
    scale_factor = scale_factors[0]

    # We want the original min_x, max_x *before* scaling. So let's do:
    # we can do min_x = min( x_dict[c]/scale_factor ) etc. if we didn't store original
    # Or store them if needed. For simplicity, let's just do what we can:
    min_x = float("inf")
    max_x = float("-inf")
    for i, xd in enumerate(x_dicts):
        fac = scale_factors[i]
        # we revert the scaling by dividing by fac to get the original coords
        xs_orig = [x/fac for x in xd.values()]
        mi, ma = min(xs_orig), max(xs_orig)
        if mi<min_x: min_x=mi
        if ma>max_x: max_x=ma

    data_range_original = max_x - min_x
    bar_len_orig = pick_scale_length(data_range_original)
    # the bar in data coords after scaling is bar_len_orig * scale_factor
    bar_len_data = bar_len_orig * scale_factor

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x_range = xlim[1]-xlim[0]
    y_range = ylim[1]-ylim[0] if ylim[1]>ylim[0] else (ylim[0]-ylim[1])
    xmarg = 0.02*x_range
    ymarg = 0.02*y_range

    if "bottom" in location:
        if ax.yaxis_inverted():
            y0 = ylim[0]+ymarg
        else:
            y0 = ylim[1]-ymarg
    else:
        # top
        if ax.yaxis_inverted():
            y0 = ylim[1]-ymarg
        else:
            y0 = ylim[0]+ymarg

    if "left" in location:
        x0 = xlim[0]+xmarg
    else:
        x0 = xlim[1]-xmarg - bar_len_data

    x1 = x0+bar_len_data
    ax.plot([x0,x1],[y0,y0], lw=line_width, color=color, zorder=10)
    # label the original bar length
    ax.text(
        0.5*(x0+x1), y0,
        f"{bar_len_orig:g}",
        ha="center",
        va="bottom" if not ax.yaxis_inverted() else "top",
        fontsize=fontsize,
        color=color,
        zorder=11
    )

#################################
# 5) DEMO
#################################

def demo():
    import Bio.Phylo as bp
    # Example newick with small branch lengths
    newick_str = "((A:1,B:2):3,(C:0.5,D:0.8):1.2);"
    tree_left = bp.read(StringIO(newick_str), "newick")
    tree_right = bp.read(StringIO(newick_str), "newick")

    fig, axes = plt.subplots(1,2, figsize=(12,6))

    # 1) Plot left tree with scale_factor=50 (for example)
    xdict_left, ydict_left, scale_factor_left = plot_tree(
        tree_left,
        ax=axes[0],
        flip_x=False,
        scale_factor=50.0,  # scale up small branch lengths by 50
        line_color="black",
        line_width=2.0
    )
    all_left = list(tree_left.find_clades())

    # 2) Add uniform triangles on the right side, labels outside
    add_uniform_triangles(
        ax=axes[0],
        x_dict=xdict_left,
        y_dict=ydict_left,
        all_clades=all_left,
        side="right",
        triangle_offset=5.0,  # bigger offset in data coords
        triangle_color="green",
        triangle_alpha=0.8,
        # cluster_labels=["C1","C2","C3","C4"],
        label_fontsize=10
    )
    axes[0].set_title("Left: Normal Tree, scale_factor=50, Triangles on Right", fontsize=11)

    # 3) Plot right tree with flip_x=True, same scale factor
    xdict_right, ydict_right, scale_factor_right = plot_tree(
        tree_right,
        ax=axes[1],
        flip_x=True,
        scale_factor=50.0,  # same scale up
        line_color="black",
        line_width=2.0
    )
    all_right = list(tree_right.find_clades())

    add_uniform_triangles(
        ax=axes[1],
        x_dict=xdict_right,
        y_dict=ydict_right,
        all_clades=all_right,
        side="left",
        triangle_offset=5.0,
        triangle_color="orange",
        triangle_alpha=0.8,
        # cluster_labels=["C1","C2","C3","C4"],
        label_fontsize=10
    )
    axes[1].set_title("Right: Flipped Tree, scale_factor=50, Triangles on Left", fontsize=11)

    # 4) Single scale bar in the left subplot
    # We'll pass the x_dicts plus their scale_factors so we can revert to original coords
    add_scale_bar(
        ax=axes[0],
        x_dicts=[xdict_left, xdict_right],
        scale_factors=[scale_factor_left, scale_factor_right],
        location="bottomleft",
        line_width=2.0,
        color="black",
        fontsize=9
    )

    plt.tight_layout()
    plt.show()

if __name__=="__main__":
    demo()

In [ ]:
# Identify the MRCAs for each cluster
mrca_1 = tree.common_ancestor(["A", "B"])
mrca_2 = tree.common_ancestor(["D", "C"])
mrca_3 = tree.common_ancestor(["E", "F", "G", "H"])

plot_tree(
    tree,
    clade_color_map=clade_color_map,
    line_width=3,
    height_scale=2,
    width_scale=1,
    hide_internal_nodes=False,  # ensure internal nodes (like MRCAs) are visible
    show_terminal_labels=False,
    label_func=lambda x: "",
    cluster_mrcas=[mrca_1, mrca_2, mrca_3],
    cluster_labels=["Cluster 1", "Cluster 2", "Cluster 3"],
)
plt.show()  # Display the figure

In [ ]:
from Bio import Phylo

# Assume 'tree' is your Bio.Phylo tree object
# Define your clusters as lists of leaf names
clusters = [
    ["A", "B"],  # Cluster 1
    ["D", "C"],  # Cluster 2
    ["E","F", "G", "H"],  # Cluster 3
]

# Assign colors for each cluster
cluster_colors = ["red", "blue", "green"]

# Create a map from clade to color
clade_color_map = {}

for i, cluster in enumerate(clusters):
    color = cluster_colors[i]
    mrca = tree.common_ancestor(cluster)

    # Color the entire subtree under this MRCA
    for descendant in mrca.find_clades():
        clade_color_map[descendant] = color
    if mrca in clade_color_map:
        del clade_color_map[mrca]
plot_tree(
    tree,
    clade_color_map=clade_color_map,
    line_width=3,
    height_scale=2,
    width_scale=1,
    hide_internal_nodes=True,
    show_terminal_labels=False,
    label_func=lambda x: "",
)

In [ ]:
def plot_tree(
    input_tree: Any,
    label_func: Optional[Callable[[Any], str]] = None,
    title: str = "",
    ax: Optional[plt.Axes] = None,
    output_name: Optional[str] = None,
    outgroup: Optional[str] = None,
    width_scale: float = 1,
    height_scale: float = 1,
    show_terminal_labels: bool = False,
    show_branch_lengths: bool = True,
    show_branch_support: bool = False,
    show_events: bool = False,
    branch_labels: Optional[Union[Dict[Any, str], Callable[[Any], str]]] = None,
    label_colors: Optional[Union[Dict[str, str], Callable[[str], str]]] = None,
    hide_internal_nodes: bool = False,
    marker_size: Optional[int] = None,
    line_width: Optional[float] = None,
    clade_color_map: Optional[Dict[Any, str]] = None,  # <--- New parameter
    **kwargs: Any,
) -> plt.Figure:
    """Plot the given tree using matplotlib.

    ...
    Added: clade_color_map : optional dict of {clade: color_str}
        If provided, branches leading to the given clade will be drawn in the specified color.
    """
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        try:
            import pylab as plt
        except ImportError:
            raise MEDICCPlotError(
                "Install matplotlib or pylab if you want to use draw."
            ) from None

    import matplotlib.collections as mpcollections

    marker_size = marker_size or SIZES["tree_marker"]
    line_width = line_width or LINEWIDTHS.get("segment_boundary", 1.0)
    label_func = label_func or (lambda x: x.name if hasattr(x, "name") else x)

    horizontal_lines = []
    vertical_lines = []
    horizontal_colors = []
    vertical_colors = []
    horizontal_lws = []
    vertical_lws = []

    marker_x = []
    marker_y = []
    marker_sizes = []
    marker_colors = []

    text_x = []
    text_y = []
    texts = []
    text_colors = []

    if ax is None:
        nsamp = len(list(input_tree.find_clades()))
        plot_height = height_scale * nsamp * 0.25
        max_leaf_to_root_distances = np.max(
            [
                np.sum([x.branch_length for x in input_tree.get_path(leaf)])
                for leaf in input_tree.get_terminals()
            ]
        )
        # plot_width = 5 + np.max(
        #     [0, width_scale * np.log10(max_leaf_to_root_distances / 100) * 5]
        # )
        plot_width = 5 + np.max([0, width_scale * 0.5])
        # maximum figure size is 250x250 inches
        fig, ax = plt.subplots(figsize=(min(250, plot_width), min(250, plot_height)))

    get_label_color = lambda label: "black"
    if label_colors is not None:
        get_label_color = (
            label_colors
            if callable(label_colors)
            else lambda label: label_colors.get(label, "black")
        )
    else:
        clade_colors = {}
        for clade in input_tree.find_clades():
            sample = clade.name
            if sample is None:
                continue
            is_terminal = clade.is_terminal()
            clade_colors[sample] = (
                COLORS["marker_terminal"] if is_terminal else COLORS["marker_internal"]
            )
            if sample == outgroup:
                clade_colors[sample] = COLORS["marker_normal"]
        get_label_color = lambda label: clade_colors.get(label, "black")

    marker_size = marker_size if marker_size is not None else SIZES["tree_marker"]
    marker_func = lambda x: (
        (marker_size, get_label_color(x.name)) if x.name is not None else None
    )

    ax.axes.get_yaxis().set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True, prune=None))
    ax.xaxis.set_tick_params(labelsize=SIZES["xlabel_tick"])
    ax.xaxis.label.set_size(SIZES["xlabel_font"])
    ax.set_title(
        title,
        x=0.01,
        y=1.0,
        ha="left",
        va="bottom",
        fontweight="bold",
        fontsize=16,
        zorder=10,
    )
    x_posns = _get_x_positions(input_tree)
    y_posns = _get_y_positions(
        input_tree, adjust=not hide_internal_nodes, outgroup=outgroup
    )
    xmax = max(x_posns.values())
    ax.set_xlim(-0.05 * xmax, 1.05 * xmax)
    top_margin = 0.5
    ymax = max(y_posns.values()) + top_margin
    ax.set_ylim(ymax, -0.5)
    ax_scale = ax.get_xlim()[1] - ax.get_xlim()[0]

    def value_to_str(value):
        if value is None or value == 0:
            return None
        return str(int(value)) if int(value) == value else str(value)

    if not branch_labels:
        if show_branch_lengths:

            def format_branch_label(x):
                return (
                    value_to_str(np.round(x.branch_length, 1))
                    if x.name != "root" and x.name is not None
                    else None
                )

        else:

            def format_branch_label(clade):
                return None

    elif isinstance(branch_labels, dict):

        def format_branch_label(clade):
            return branch_labels.get(clade)

    else:
        if not callable(branch_labels):
            raise TypeError(
                "branch_labels must be either a dict or a callable (function)"
            )

        def format_branch_label(clade):
            return value_to_str(branch_labels(clade))

    if show_branch_support:

        def format_support_value(clade):
            if clade.name == "root" or clade.name is None:
                return None
            try:
                confidences = clade.confidences
            except AttributeError:
                pass
            else:
                return "/".join(value_to_str(cnf.value) for cnf in confidences)
            return (
                value_to_str(clade.confidence) if clade.confidence is not None else None
            )

    def draw_clade_lines(
        use_linecollection: bool = False,
        orientation: str = "horizontal",
        y_here: float = 0,
        x_start: float = 0,
        x_here: float = 0,
        y_bot: float = 0,
        y_top: float = 0,
        color: str = "black",
        lw: float = 0.1,
    ) -> None:
        if use_linecollection and orientation == "horizontal":
            horizontal_lines.append([(x_start, y_here), (x_here, y_here)])
            horizontal_colors.append(color)
            horizontal_lws.append(lw)
        elif use_linecollection and orientation == "vertical":
            vertical_lines.append([(x_here, y_bot), (x_here, y_top)])
            vertical_colors.append(color)
            vertical_lws.append(lw)

    def draw_clade(clade: Any, x_start: float, default_color: str, lw: float) -> None:
        x_here = x_posns[clade]
        y_here = y_posns[clade]

        # Determine line color for this clade (horizontal line to this node)
        line_color = (
            clade_color_map.get(clade, default_color)
            if clade_color_map
            else default_color
        )

        if hasattr(clade, "color") and clade.color is not None:
            line_color = clade.color.to_hex()
        if hasattr(clade, "width") and clade.width is not None:
            lw = clade.width * plt.rcParams["lines.linewidth"]

        # Draw horizontal line to this clade
        draw_clade_lines(
            use_linecollection=True,
            orientation="horizontal",
            y_here=y_here,
            x_start=x_start,
            x_here=x_here,
            color=line_color,
            lw=lw,
        )

        if marker_func is not None:
            marker = marker_func(clade)
            if (
                marker is not None
                and clade is not None
                and not (hide_internal_nodes and not clade.is_terminal())
            ):
                m_size, m_color = marker
                marker_x.append(x_here)
                marker_y.append(y_here)
                marker_sizes.append(m_size)
                marker_colors.append(m_color)

        label = label_func(str(clade.name))
        if label not in (None, clade.__class__.__name__) and not (
            hide_internal_nodes and not clade.is_terminal()
        ):
            text_x.append(x_here + min(0.02 * ax_scale, 1))
            text_y.append(y_here)
            texts.append(f" {label}")
            text_colors.append(get_label_color(label))

        if clade.clades:
            y_top_local = y_posns[clade.clades[0]]
            y_bot_local = y_posns[clade.clades[-1]]

            # Determine vertical line color based on children's colors
            child_colors = []
            for child in clade.clades:
                # Child color is looked up; if not found, fallback to parent's line_color
                child_line_color = (
                    clade_color_map.get(child, line_color)
                    if clade_color_map
                    else line_color
                )
                child_colors.append(child_line_color)

            # If all children share the same color, use that for the vertical line
            if len(set(child_colors)) == 1:
                vertical_line_color = child_colors[0]
            else:
                vertical_line_color = line_color

            # Draw vertical line with computed vertical_line_color
            draw_clade_lines(
                use_linecollection=True,
                orientation="vertical",
                x_here=x_here,
                y_bot=y_bot_local,
                y_top=y_top_local,
                color=vertical_line_color,
                lw=lw,
            )

            # Recurse into children, passing the parent's line_color as default
            for child in clade:
                draw_clade(child, x_here, line_color, lw)

    line_width = (
        line_width if line_width is not None else plt.rcParams["lines.linewidth"]
    )
    draw_clade(input_tree.root, 0, "k", line_width)

    if horizontal_lines:
        h_linecollection = mpcollections.LineCollection(
            horizontal_lines,
            colors=horizontal_colors,
            linewidths=horizontal_lws,
        )
        ax.add_collection(h_linecollection)

    if vertical_lines:
        v_linecollection = mpcollections.LineCollection(
            vertical_lines,
            colors=vertical_colors,
            linewidths=vertical_lws,
        )
        ax.add_collection(v_linecollection)

    if marker_x:
        ax.scatter(marker_x, marker_y, s=marker_sizes, c=marker_colors, zorder=3)

    for x, y, text, color in zip(text_x, text_y, texts, text_colors):
        ax.text(x, y, text, verticalalignment="center", color=color)

    ax.set_xlabel("branch length")
    ax.set_ylabel("taxa")

    for key, value in kwargs.items():
        try:
            list(value)
        except TypeError:
            raise ValueError(
                f'Keyword argument "{key}={value}" is not in the correct format.'
            ) from None

        if isinstance(value, dict):
            getattr(plt, str(key))(**dict(value))
        elif not (isinstance(value[0], tuple)):
            getattr(plt, str(key))(*value)
        elif isinstance(value[0], tuple):
            getattr(plt, str(key))(*value[0], **dict(value[1]))

    if output_name is not None:
        plt.savefig(output_name + ".png", bbox_inches="tight")

    return plt.gcf()

# Load bird tree and the two clusters. Prune tree such that both trees have their nodes in

In [ ]:
from phytclust.main import PhytClust
from phytclust import plot_tree, plot_multiple_clusters
from phytclust import pairwise_distances
import io
import os
from Bio import Phylo

tree_path = "/home/ganesank/project/phytclust/Data/Species/Avian_time_tree.newick"
tree = Phylo.read(tree_path, "newick", rooted=True)

In [ ]:
clust_obj = PhytClust(tree, should_plot_scores=True, num_peaks=10, resolution_on=True)

In [ ]:
import pandas as pd
from Bio.Phylo.BaseTree import Clade
tree = clust_obj.tree
file_path = "/home/ganesank/project/phytclust/Data/Species/avian_list_2024.txt"
df = pd.read_csv(file_path, sep="\t")

orders = df["order"].unique()

order_to_group = {
    # Palaeognathae
    "Struthioniformes": "Palaeognathae",
    "Rheiformes": "Palaeognathae",
    "Casuariiformes": "Palaeognathae",
    "Apterygiformes": "Palaeognathae",
    "Tinamiformes": "Palaeognathae",
    # Galloanseres
    "Galliformes": "Galloanseres",
    "Anseriformes": "Galloanseres",
}


def map_to_group(order):
    return order_to_group.get(order, "Neoaves")


df["group"] = df["order"].apply(map_to_group)

tree_clades = [
    Clade(branch_length=terminal.branch_length, name=terminal.name)
    for terminal in tree.get_terminals()
]

group_to_cluster = {"Neoaves": 2, "Galloanseres": 1, "Palaeognathae": 0}
df["cluster"] = df["group"].map(group_to_cluster)

species_to_cluster = dict(zip(df["taxa"], df["cluster"]))

clusters_low = {
    clade: species_to_cluster.get(clade.name, None)
    for clade in tree_clades
    if clade.name in species_to_cluster
}

import pandas as pd
from Bio.Phylo.BaseTree import Clade

mag7 = df["clade_mag7"].unique()

tree_clades = [
    Clade(branch_length=terminal.branch_length, name=terminal.name)
    for terminal in tree.get_terminals()
]

group_to_cluster = {
    "Telluraves": 6,
    "Strisores": 3,
    "Cursorimorphae": 2,
    "Galloanseres": 1,
    "Columbimorphae": 4,
    "Aequornithes": 5,
    "Palaeognathae": 0,
    "Otidimorphae": 7,
    "Phaethontimorphae": 8,
    "Opisthocomiformes": 9,
    "Mirandornithes": 10,
}
df["clusters_high"] = df["clade_mag7"].map(group_to_cluster)

species_to_cluster = dict(zip(df["taxa"], df["clusters_high"]))

clusters_high = {
    clade: species_to_cluster.get(clade.name, None)
    for clade in tree_clades
    if clade.name in species_to_cluster
}

# print(clade_to_cluster)

# Filter the DataFrame for rows where 'clade_mag7' is 'Telluraves'
filtered_df = df[df["clade_mag7"] == "Telluraves"]

# Get the unique values in the 'family' column for the filtered DataFrame
unique_families = filtered_df["order"].unique()

# Print the unique families
print(unique_families)
import pandas as pd
from Bio.Phylo.BaseTree import Clade

# Assuming df and tree are already defined

# Step 1: Filter the DataFrame for 'Telluraves'
filtered_df = df[df["clade_mag7"] == "Telluraves"]

# Step 2: Map the 'order' column to 'Australaves' and 'Afroaves'
telluraves_mapping = {
    "Passeriformes": "Australaves",
    "Psittaciformes": "Australaves",
    "Trogoniformes": "Afroaves",
    "Accipitriformes": "Afroaves",
    "Coraciiformes": "Afroaves",
    "Piciformes": "Afroaves",
    "Bucerotiformes": "Afroaves",
    "Cariamiformes": "Australaves",
    "Strigiformes": "Afroaves",
    "Coliiformes": "Afroaves",
    "Falconiformes": "Australaves",
    "Leptosomatiformes": "Afroaves",
}

filtered_df["telluraves_group"] = filtered_df["order"].map(telluraves_mapping)

# Step 3: Update the group_to_cluster dictionary
group_to_cluster = {
    "Palaeognathae": 0,
    "Galloanseres": 1,
    "Mirandornithes": 2,
    "Columbimorphae": 3,
    "Otidimorphae": 4,
    "Opisthocomiformes": 5,
    "Cursorimorphae": 6,
    "Strisores": 7,
    "Phaethontimorphae": 8,
    "Aequornithes": 9,
    "Australaves": 10,
    "Afroaves": 11,
    "Telluraves": 12,
}

# Step 4: Map the species to clusters
df["clusters_high"] = df["clade_mag7"].map(group_to_cluster)
df.loc[df["clade_mag7"] == "Telluraves", "clusters_high"] = filtered_df[
    "telluraves_group"
].map(group_to_cluster)

species_to_cluster = dict(zip(df["taxa"], df["clusters_high"]))

# Step 5: Update the clusters_high dictionary
tree_clades = [
    Clade(branch_length=terminal.branch_length, name=terminal.name)
    for terminal in tree.get_terminals()
]

clusters_high = {
    clade: species_to_cluster.get(clade.name, None)
    for clade in tree_clades
    if clade.name in species_to_cluster
}

# Print the clusters_high dictionary
print(clusters_high)

In [ ]:
clust_obj.plot(n=13)

In [ ]:
import pandas as pd

clust_obj_clusters = [
    clusters_high,
    # clust_obj.clusters[11],
    # clusters_low,  # from paper - 3 families
    # clust_obj.clusters[clust_obj.best_peaks[5] - 1],  # clusters low - phytclust
    # family 13
    # clust_obj.clusters[clust_obj.best_peaks[6] - 1],
]
comparison_ids = ["6", "8", "20", "21"]
rows = []

for clade_dict, comparison_id in zip(clust_obj_clusters, comparison_ids):
    for clade, cluster_id in clade_dict.items():
        leaf_name = clade.name
        rows.append([leaf_name, comparison_id, cluster_id])

df_clusters = pd.DataFrame(rows, columns=["leaf_name", "comparison_IDs", "cluster_ID"])
df_clusters.set_index(["leaf_name", "comparison_IDs"], inplace=True)
import pandas as pd
import matplotlib.pyplot as plt
from Bio import Phylo


# Call the plot_multiple_clusters function with the custom colormaps
fig = plot_multiple_clusters(input_df=df_clusters, final_tree=tree, outgroup=None)

# Show the plot
plt.show()

In [ ]:
from collections import defaultdict
# tree = clust_obj.tree
# clusters_high
cluster2leaves = defaultdict(list)
for leaf, cluster_id in clusters_high.items():
    cluster2leaves[cluster_id].append(leaf.name)
mrcas_1 = []
for cluster_id, leaves in cluster2leaves.items():
    if len(leaves) == 1:
        mrcas_1.append(tree.find_any(name=leaves[0]))
    else:
        mrcas_1.append(tree.common_ancestor(leaves))

cluster2leaves = defaultdict(list)
for leaf, cluster_id in clust_obj.clusters[clust_obj.best_peaks[1] - 1].items():
    cluster2leaves[cluster_id].append(leaf.name)
mrcas_2 = []
for cluster_id, leaves in cluster2leaves.items():
    if len(leaves) == 1:
        mrcas_2.append(tree.find_any(name=leaves[0]))
    else:
        mrcas_2.append(tree.common_ancestor(leaves))

In [ ]:
def is_ancestor(tree, ancestor_clade, descendant_clade):
    """
    Returns True if ancestor_clade is an ancestor of descendant_clade in the given tree.
    """
    path_to_descendant = tree.get_path(descendant_clade)
    return ancestor_clade in path_to_descendant[:-1]

set_1 = set(mrcas_1)
set_2 = set(mrcas_2)

# Intersection: MRCAs appearing in both sets
common_mrcas = set_1.intersection(set_2)

# Unique in each set
unique_1 = set_1 - common_mrcas
unique_2 = set_2 - common_mrcas

# Combine all candidates
candidates = list(common_mrcas) + list(unique_1) + list(unique_2)


def get_depth(tree, clade):
    return len(tree.get_path(clade))


candidates = sorted(candidates, key=lambda c: get_depth(tree, c))

final_mrcas = []

for candidate in candidates:
    if any(is_ancestor(tree, candidate, fm) for fm in final_mrcas):
        continue

    ancestor_in_final = [fm for fm in final_mrcas if is_ancestor(tree, fm, candidate)]
    if ancestor_in_final:
        for anc in ancestor_in_final:
            final_mrcas.remove(anc)
        final_mrcas.append(candidate)
    else:
        final_mrcas.append(candidate)


print("Final MRCAs:", final_mrcas)

In [ ]:
# def collect_descendants(tree, mrca_list):
#     descendants = set()
#     for mrca in mrca_list:
#         for clade in mrca.find_clades():
#             if clade != mrca:
#                 descendants.add(clade)
#     return descendants


# def prune_tree_to_mrcas(tree, final_mrcas):
#     descendants_to_prune = collect_descendants(tree, final_mrcas)
#     for clade in descendants_to_prune:
#         try:
#             tree.prune(clade)
#         except ValueError:
#             continue
#     return tree


def prune_tree_to_mrcas(tree, final_mrcas):
    # Collapse each MRCA to remove all its descendants.
    for mrca in final_mrcas:
        tree.collapse(mrca)  # This makes mrca a leaf by removing its descendants.

    # Now remove leaves that are not in final_mrcas
    to_remove = [leaf for leaf in tree.get_terminals() if leaf not in final_mrcas]
    for leaf in to_remove:
        tree.prune(leaf)

    return tree


to_prune = deepcopy(tree)
prune_tree_to_mrcas(to_prune, final_mrcas)

# # Plot the pruned tree
Phylo.draw(to_prune)

In [ ]:
final_mrcas

In [ ]:
from Bio import Phylo


def prune_tree_to_mrcas(tree, final_mrcas):

    resolved_mrcas = []
    for mrca in final_mrcas:
        if mrca.name is None:
            raise ValueError(
                "MRCA does not have a name. We need a name to match it in the tree."
            )

        found = next(tree.find_clades(name=mrca.name), None)
        if not found:
            raise ValueError(
                f"MRCA with name '{mrca.name}' not found in the current tree."
            )
        resolved_mrcas.append(found)

    for mrca_clade in resolved_mrcas:
        if not mrca_clade.is_terminal():
            tree.collapse(mrca_clade)

    mrca_names = {mrca.name for mrca in resolved_mrcas}

    # Prune any terminal leaf that isn't one of the MRCA nodes we just collapsed
    to_remove = [leaf for leaf in tree.get_terminals() if leaf.name not in mrca_names]
    for leaf in to_remove:
        tree.prune(leaf)

    return tree

In [ ]:
from Bio import Phylo


def prune_descendants_not_in_set(clade, keep_names):
    """
    Recursively prune all descendants of 'clade' that do not have names in 'keep_names'.
    If a descendant is terminal and not in keep_names, remove it.
    If a descendant is internal and, after pruning its children, has no MRCA descendants, remove it.
    Returns True if this clade (or its descendants) contains at least one MRCA name, False otherwise.
    """
    # If this is a leaf:
    if clade.is_terminal():
        return clade.name in keep_names

    new_children = []
    for child in clade.clades:
        if prune_descendants_not_in_set(child, keep_names):
            new_children.append(child)

    clade.clades = new_children
    if not clade.clades and (clade.name not in keep_names):
        return False

    return True


def prune_tree_to_mrcas(tree, final_mrcas):
    resolved_mrcas = []
    for mrca in final_mrcas:
        if mrca.name is None:
            raise ValueError(
                "MRCA does not have a name. A name is required to match it in the tree."
            )

        found = next(tree.find_clades(name=mrca.name), None)
        if not found:
            raise ValueError(
                f"MRCA with name '{mrca.name}' not found in the current tree."
            )
        resolved_mrcas.append(found)

    mrca_names = {mrca.name for mrca in resolved_mrcas}
    for mrca_clade in resolved_mrcas:
        prune_descendants_not_in_set(mrca_clade, mrca_names)
    prune_descendants_not_in_set(tree.root, mrca_names)

    return tree

In [ ]:
to_prune = deepcopy(tree)
prune_tree_to_mrcas(to_prune, final_mrcas)

# # Plot the pruned tree
Phylo.draw(to_prune)

In [ ]:
mrcas_2

In [ ]:
mrcas_1

In [ ]:
Phylo.draw(to_prune)

In [ ]:
def collect_terminal_nodes(mrca, tree):
    for clade in tree.find_clades():
        if clade.name == mrca.name:
            if clade.is_terminal():
                return [clade]
            else:
                return clade.get_terminals()


# Create a dictionary for MRCA 1
mrca_dict_1 = {}
for idx, mrca in enumerate(mrcas_1):
    terminal_nodes = collect_terminal_nodes(mrca, to_prune)
    for terminal in terminal_nodes:
        mrca_dict_1[terminal] = idx

# Create a dictionary for MRCA 2
mrca_dict_2 = {}
for idx, mrca in enumerate(mrcas_2):
    terminal_nodes = collect_terminal_nodes(mrca, to_prune)
    for terminal in terminal_nodes:
        mrca_dict_2[terminal] = idx

import pandas as pd

clust_obj_clusters = [
    mrca_dict_1,
    mrca_dict_2,
]
comparison_ids = ["6", "8"]
rows = []

for clade_dict, comparison_id in zip(clust_obj_clusters, comparison_ids):
    for clade, cluster_id in clade_dict.items():
        leaf_name = clade.name
        rows.append([leaf_name, comparison_id, cluster_id])

df_clusters = pd.DataFrame(rows, columns=["leaf_name", "comparison_IDs", "cluster_ID"])
df_clusters.set_index(["leaf_name", "comparison_IDs"], inplace=True)
import pandas as pd
import matplotlib.pyplot as plt
from Bio import Phylo


# Call the plot_multiple_clusters function with the custom colormaps
fig = plot_multiple_clusters(input_df=df_clusters, final_tree=to_prune, outgroup=None)

# Show the plot
plt.show()

In [ ]:
fig = plot_multiple_clusters(input_df=df_clusters, final_tree=to_prune, outgroup=None, tree_width_ratio=5)

In [ ]:
mrca_dict_1

In [ ]:
from typing import Any, Callable, Optional, Union, Dict
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from Bio.Phylo.BaseTree import Clade
import matplotlib.collections as mpcollections

# You should define SIZES, COLORS, and LINEWIDTHS as used in your code.
# For this example, let's add some dummy definitions:
SIZES = {
    "tree_marker": 50,
    "xlabel_tick": 10,
    "xlabel_font": 12,
}
COLORS = {
    "marker_terminal": "black",
    "marker_internal": "black",
    "marker_normal": "black",
}
LINEWIDTHS = {"segment_boundary": 1.0}


def _get_x_positions(tree: Any) -> Dict[Any, float]:
    """Create a mapping of each clade to its horizontal position.
    Dict of {clade: x-coord}
    """
    depths = tree.depths()
    # If there are no branch lengths, assume unit branch lengths
    if not max(depths.values()):
        depths = tree.depths(unit_branch_lengths=True)
    return depths


def _get_y_positions(
    tree: Any, adjust: bool = False, outgroup: str = "outgroup"
) -> Dict[Any, float]:
    """Create a mapping of each clade to its vertical position.
        Dict of {clade: y-coord}.
    Coordinates are negative, and integers for tips.
    """
    maxheight = tree.count_terminals()
    heights = {
        tip: maxheight - 1 - i
        for i, tip in enumerate(
            reversed(
                [
                    x
                    for x in tree.get_terminals()
                    if outgroup is None or x.name != outgroup
                ]
            )
        )
    }
    if outgroup is not None:
        normal_clades = list(tree.find_clades(outgroup))
        if not normal_clades:
            raise PlotError(f"Normal clade {outgroup} not found in tree")
        heights[normal_clades[0]] = maxheight

    # Internal nodes: place at midpoint of children
    def calc_row(clade: Any) -> None:
        for subclade in clade:
            if subclade not in heights:
                calc_row(subclade)
        # Closure over heights
        heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0

    if tree.root.clades:
        calc_row(tree.root)

    if adjust:
        pos = pd.DataFrame(
            [(clade, val) for clade, val in heights.items()], columns=["clade", "pos"]
        ).sort_values("pos")
        pos["newpos"] = 0
        count = 0
        for i in pos.index:
            if pos.loc[i, "clade"] != tree.root:
                count += 1
            pos.loc[i, "newpos"] = count

        pos.set_index("clade", inplace=True)
        heights = pos.to_dict()["newpos"]

    return heights


def plot_tree(
    input_tree: Any,
    label_func: Optional[Callable[[Any], str]] = None,
    title: str = "",
    ax: Optional[plt.Axes] = None,
    output_name: Optional[str] = None,
    outgroup: Optional[str] = None,
    width_scale: float = 1,
    height_scale: float = 1,
    show_terminal_labels: bool = False,
    show_branch_lengths: bool = True,
    show_branch_support: bool = False,
    show_events: bool = False,
    branch_labels: Optional[Union[Dict[Any, str], Callable[[Any], str]]] = None,
    label_colors: Optional[Union[Dict[str, str], Callable[[str], str]]] = None,
    hide_internal_nodes: bool = False,
    marker_size: Optional[int] = None,
    line_width: Optional[float] = None,
    cluster_assignments: Optional[Dict[Clade, int]] = None,
    cluster_colors: Optional[Dict[int, str]] = None,
    highlight_clusters: bool = True,
    **kwargs: Any,
) -> plt.Figure:
    """Plot the given tree using matplotlib and highlight clusters with hourglass-like brackets."""

    try:
        import matplotlib.pyplot as plt
    except ImportError:
        try:
            import pylab as plt
        except ImportError:
            raise ImportError("Install matplotlib or pylab if you want to use draw.")

    import matplotlib.collections as mpcollections

    marker_size = marker_size or SIZES["tree_marker"]
    line_width = line_width or LINEWIDTHS.get("segment_boundary", 1.0)
    # label_func = label_func or (lambda x: x.name if hasattr(x, "name") else x)
    label_func = label_func or (lambda x: "" if hasattr(x, "name") else "")

    horizontal_lines = []
    vertical_lines = []
    horizontal_colors = []
    vertical_colors = []
    horizontal_lws = []
    vertical_lws = []

    marker_x = []
    marker_y = []
    marker_sizes = []
    marker_colors = []

    text_x = []
    text_y = []
    texts = []
    text_colors = []

    # Setup figure if not given
    if ax is None:
        nsamp = len(list(input_tree.find_clades()))
        plot_height = height_scale * nsamp * 0.25
        max_leaf_to_root_distances = np.max(
            [
                np.sum([x.branch_length for x in input_tree.get_path(leaf)])
                for leaf in input_tree.get_terminals()
            ]
        )
        plot_width = 5 + np.max(
            [0, width_scale * np.log10(max_leaf_to_root_distances / 100) * 5]
        )
        fig, ax = plt.subplots(figsize=(min(250, plot_width), min(250, plot_height)))
    else:
        fig = plt.gcf()

    get_label_color = lambda label: "black"

    if label_colors is not None:
        get_label_color = (
            label_colors
            if callable(label_colors)
            else lambda label: label_colors.get(label, "black")
        )
    else:
        clade_colors = {}
        for clade in input_tree.find_clades():
            sample = clade.name
            if sample is None:
                continue
            is_terminal = clade.is_terminal()
            clade_colors[sample] = (
                COLORS["marker_terminal"] if is_terminal else COLORS["marker_internal"]
            )
            if sample == outgroup:
                clade_colors[sample] = COLORS["marker_normal"]
        get_label_color = lambda label: clade_colors.get(label, "black")

    marker_size = marker_size if marker_size is not None else SIZES["tree_marker"]
    marker_func = lambda x: (
        (marker_size, get_label_color(x.name)) if x.name is not None else None
    )

    ax.axes.get_yaxis().set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True, prune=None))
    ax.xaxis.set_tick_params(labelsize=SIZES["xlabel_tick"])
    ax.xaxis.label.set_size(SIZES["xlabel_font"])
    ax.set_title(
        title,
        x=0.01,
        y=1.0,
        ha="left",
        va="bottom",
        fontweight="bold",
        fontsize=16,
        zorder=10,
    )

    x_posns = _get_x_positions(input_tree)
    y_posns = _get_y_positions(
        input_tree, adjust=not hide_internal_nodes, outgroup=outgroup
    )
    xmax = max(x_posns.values())
    ax.set_xlim(-0.05 * xmax, 1.05 * xmax)
    top_margin = 0.5
    ymax = max(y_posns.values()) + top_margin
    ax.set_ylim(ymax, -0.5)
    ax_scale = ax.get_xlim()[1] - ax.get_xlim()[0]

    def value_to_str(value):
        if value is None or value == 0:
            return None
        return str(int(value)) if int(value) == value else str(value)

    if not branch_labels:
        if show_branch_lengths:

            def format_branch_label(x):
                return (
                    value_to_str(np.round(x.branch_length, 1))
                    if x.name != "root" and x.name is not None
                    else None
                )

        else:

            def format_branch_label(clade):
                return None

    elif isinstance(branch_labels, dict):

        def format_branch_label(clade):
            return branch_labels.get(clade)

    else:
        if not callable(branch_labels):
            raise TypeError(
                "branch_labels must be either a dict or a callable (function)"
            )

        def format_branch_label(clade):
            return value_to_str(branch_labels(clade))

    if show_branch_support:

        def format_support_value(clade):
            if clade.name == "root" or clade.name is None:
                return None
            try:
                confidences = clade.confidences
            except AttributeError:
                pass
            else:
                return "/".join(value_to_str(cnf.value) for cnf in confidences)
            return (
                value_to_str(clade.confidence) if clade.confidence is not None else None
            )

    def draw_clade_lines(
        use_linecollection: bool = False,
        orientation: str = "horizontal",
        y_here: float = 0,
        x_start: float = 0,
        x_here: float = 0,
        y_bot: float = 0,
        y_top: float = 0,
        color: str = "black",
        lw: float = 0.1,
    ) -> None:
        if use_linecollection and orientation == "horizontal":
            horizontal_lines.append([(x_start, y_here), (x_here, y_here)])
            horizontal_colors.append(color)
            horizontal_lws.append(lw)
        elif use_linecollection and orientation == "vertical":
            vertical_lines.append([(x_here, y_bot), (x_here, y_top)])
            vertical_colors.append(color)
            vertical_lws.append(lw)

    def draw_clade(clade: Any, x_start: float, color: str, lw: float) -> None:
        x_here = x_posns[clade]
        y_here = y_posns[clade]
        if hasattr(clade, "color") and clade.color is not None:
            color = clade.color.to_hex()
        if hasattr(clade, "width") and clade.width is not None:
            lw = clade.width * plt.rcParams["lines.linewidth"]

        draw_clade_lines(
            use_linecollection=True,
            orientation="horizontal",
            y_here=y_here,
            x_start=x_start,
            x_here=x_here,
            color=color,
            lw=lw,
        )

        if marker_func is not None:
            marker = marker_func(clade)
            if (
                marker is not None
                and clade is not None
                and not (hide_internal_nodes and not clade.is_terminal())
            ):
                m_size, m_color = marker
                marker_x.append(x_here)
                marker_y.append(y_here)
                marker_sizes.append(m_size)
                marker_colors.append(m_color)

        label = label_func(str(clade.name))
        if label not in (None, clade.__class__.__name__) and not (
            hide_internal_nodes and not clade.is_terminal()
        ):
            text_x.append(x_here + min(0.02 * ax_scale, 1))
            text_y.append(y_here)
            texts.append(f" {label}")
            text_colors.append(get_label_color(label))

        if clade.clades:
            y_top = y_posns[clade.clades[0]]
            y_bot = y_posns[clade.clades[-1]]
            draw_clade_lines(
                use_linecollection=True,
                orientation="vertical",
                x_here=x_here,
                y_bot=y_bot,
                y_top=y_top,
                color=color,
                lw=lw,
            )
            for child in clade:
                draw_clade(child, x_here, color, lw)

    line_width = (
        line_width if line_width is not None else plt.rcParams["lines.linewidth"]
    )
    draw_clade(input_tree.root, 0, "k", line_width)

    # Add line collections
    if horizontal_lines:
        h_linecollection = mpcollections.LineCollection(
            horizontal_lines,
            colors=horizontal_colors,
            linewidths=horizontal_lws,
        )
        ax.add_collection(h_linecollection)

    if vertical_lines:
        v_linecollection = mpcollections.LineCollection(
            vertical_lines,
            colors=vertical_colors,
            linewidths=vertical_lws,
        )
        ax.add_collection(v_linecollection)

    # Plot markers
    if marker_x:
        ax.scatter(marker_x, marker_y, s=marker_sizes, c=marker_colors, zorder=3)

    # Plot texts
    for xx, yy, text, color in zip(text_x, text_y, texts, text_colors):
        ax.text(xx, yy, text, verticalalignment="center", color=color)

    ax.set_xlabel("branch length")
    ax.set_ylabel("taxa")

    # Apply additional pyplot options
    for key, value in kwargs.items():
        try:
            list(value)
        except TypeError:
            raise ValueError(
                f'Keyword argument "{key}={value}" is not in the correct format.'
            ) from None
        if isinstance(value, dict):
            getattr(plt, str(key))(**dict(value))
        elif not (isinstance(value[0], tuple)):
            getattr(plt, str(key))(*value)
        elif isinstance(value[0], tuple):
            getattr(plt, str(key))(*value[0], **dict(value[1]))

    # Highlight clusters with hourglass brackets if requested
    if highlight_clusters and cluster_assignments:
        # Group clades by cluster
        cluster_leaves_clades = {}
        for cl, c_id in cluster_assignments.items():
            # We only draw brackets around terminal sets or monophyletic sets.
            # Consider filtering only leaves or check monophyly if needed.
            if c_id not in cluster_leaves_clades:
                cluster_leaves_clades[c_id] = []
            cluster_leaves_clades[c_id].append(cl)

        cluster_bounds = {}
        for c_id, clades in cluster_leaves_clades.items():
            # Get y positions of these clades
            y_coords = [y_posns[c] for c in clades if c in y_posns]
            if not y_coords:
                continue
            y_min = min(y_coords)
            y_max = max(y_coords)
            cluster_bounds[c_id] = (y_min, y_max)

        # Place the brackets to the right of the tree
        x_lim = ax.get_xlim()
        x_annot = x_lim[1] + 0.1 * (x_lim[1] - x_lim[0])

        for c_id, (y_min, y_max) in cluster_bounds.items():
            y_mid = (y_min + y_max) / 2.0
            c_color = cluster_colors.get(c_id, "grey") if cluster_colors else "grey"

            # Draw "hourglass" like brackets:
            # Vertical line from y_max to y_mid
            ax.plot([x_annot, x_annot], [y_max, y_mid], color=c_color, lw=2)
            # Vertical line from y_min to y_mid
            ax.plot([x_annot, x_annot], [y_min, y_mid], color=c_color, lw=2)
            # Small horizontal line at y_mid for convergence
            ax.plot(
                [x_annot, x_annot + 0.02 * (x_lim[1] - x_lim[0])],
                [y_mid, y_mid],
                color=c_color,
                lw=2,
            )

            # Label the cluster
            ax.text(
                x_annot + 0.03 * (x_lim[1] - x_lim[0]),
                y_mid,
                f"Cluster {c_id}",
                va="center",
                ha="left",
                color=c_color,
                fontsize=12,
            )

        # Adjust x-limits to fit brackets
        ax.set_xlim(x_lim[0], x_annot + 0.2 * (x_lim[1] - x_lim[0]))

    if output_name is not None:
        plt.savefig(output_name + ".png", bbox_inches="tight")

    return plt.gcf()

In [ ]:
from typing import Any, Callable, Optional, Union, Dict
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from Bio.Phylo.BaseTree import Clade
import matplotlib.collections as mpcollections
from matplotlib.lines import Line2D
from collections import defaultdict
import pandas as pd


# Define PlotError
class PlotError(Exception):
    pass


# Define SIZES, COLORS, LINEWIDTHS
SIZES = {
    "tree_marker": 50,
    "xlabel_tick": 10,
    "xlabel_font": 12,
}
COLORS = {
    "marker_terminal": "black",
    "marker_internal": "black",
    "marker_normal": "black",
}
LINEWIDTHS = {"segment_boundary": 1.0}


def _get_x_positions(tree: Any) -> Dict[Any, float]:
    """Create a mapping of each clade to its horizontal position.
    Dict of {clade: x-coord}
    """
    depths = tree.depths()
    # If there are no branch lengths, assume unit branch lengths
    if not max(depths.values()):
        depths = tree.depths(unit_branch_lengths=True)
    return depths


def _get_y_positions(
    tree: Any, adjust: bool = False, outgroup: Optional[str] = "outgroup"
) -> Dict[Any, float]:
    """Create a mapping of each clade to its vertical position.
    Dict of {clade: y-coord}.
    Coordinates are negative, and integers for tips.
    """
    maxheight = tree.count_terminals()
    heights = {
        tip: maxheight - 1 - i
        for i, tip in enumerate(
            reversed(
                [
                    x
                    for x in tree.get_terminals()
                    if outgroup is None or x.name != outgroup
                ]
            )
        )
    }
    if outgroup is not None:
        normal_clades = list(tree.find_clades(outgroup))
        if not normal_clades:
            raise PlotError(f"Normal clade '{outgroup}' not found in tree")
        heights[normal_clades[0]] = maxheight

    # Internal nodes: place at midpoint of children
    def calc_row(clade: Any) -> None:
        for subclade in clade:
            if subclade not in heights:
                calc_row(subclade)
        # Closure over heights
        heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0

    if tree.root.clades:
        calc_row(tree.root)

    if adjust:
        pos = pd.DataFrame(
            [(clade, val) for clade, val in heights.items()], columns=["clade", "pos"]
        ).sort_values("pos")
        pos["newpos"] = 0
        count = 0
        for i in pos.index:
            if pos.loc[i, "clade"] != tree.root:
                count += 1
            pos.loc[i, "newpos"] = count

        pos.set_index("clade", inplace=True)
        heights = pos.to_dict()["newpos"]

    return heights


def plot_tree(
    input_tree: Any,
    label_func: Optional[Callable[[Any], str]] = None,
    title: str = "",
    ax: Optional[plt.Axes] = None,
    output_name: Optional[str] = None,
    outgroup: Optional[str] = None,
    width_scale: float = 1,
    height_scale: float = 1,
    show_terminal_labels: bool = False,
    show_branch_lengths: bool = True,
    show_branch_support: bool = False,
    show_events: bool = False,
    branch_labels: Optional[Union[Dict[Any, str], Callable[[Any], str]]] = None,
    label_colors: Optional[Union[Dict[str, str], Callable[[str], str]]] = None,
    hide_internal_nodes: bool = False,
    marker_size: Optional[int] = None,
    line_width: Optional[float] = None,
    cluster_assignments: Optional[Dict[Clade, int]] = None,
    cluster_colors: Optional[Dict[int, str]] = None,
    highlight_clusters: bool = True,
    **kwargs: Any,
) -> plt.Figure:
    """Plot the given tree using matplotlib and highlight clusters by coloring terminal nodes and connecting lines."""

    try:
        import matplotlib.pyplot as plt
    except ImportError:
        raise ImportError("Install matplotlib or pylab if you want to use draw.")

    import matplotlib.collections as mpcollections

    marker_size = marker_size or SIZES["tree_marker"]
    line_width = line_width or LINEWIDTHS.get("segment_boundary", 1.0)
    # Default label function: return empty string if no name
    label_func = label_func or (
        lambda x: "" if hasattr(x, "name") and x.name is None else str(x)
    )

    horizontal_lines = []
    vertical_lines = []
    horizontal_colors = []
    vertical_colors = []
    horizontal_lws = []
    vertical_lws = []

    marker_x = []
    marker_y = []
    marker_sizes = []
    marker_colors = []

    text_x = []
    text_y = []
    texts = []
    text_colors = []

    # Setup figure if not given
    if ax is None:
        nsamp = len(list(input_tree.find_clades()))
        plot_height = height_scale * nsamp * 0.25
        max_leaf_to_root_distances = np.max(
            [
                np.sum([x.branch_length for x in input_tree.get_path(leaf)])
                for leaf in input_tree.get_terminals()
            ]
        )
        plot_width = 5 + np.max(
            [0, width_scale]
        )
        fig, ax = plt.subplots(figsize=(min(250, plot_width), min(250, plot_height)))
    else:
        fig = plt.gcf()

    # Function to determine clade color, prioritizing cluster assignments for terminal nodes
    def get_clade_color(clade):
        if cluster_assignments and clade in cluster_assignments and clade.is_terminal():
            c_id = cluster_assignments[clade]
            return cluster_colors.get(c_id, "tan") if cluster_colors else "tan"
        else:
            # fallback: terminal and internal nodes are black
            if clade.is_terminal():
                return COLORS["marker_terminal"]
            else:
                return COLORS["marker_internal"]

    x_posns = _get_x_positions(input_tree)
    y_posns = _get_y_positions(
        input_tree, adjust=not hide_internal_nodes, outgroup=outgroup
    )
    if not x_posns:
        raise PlotError("No clades found in the tree for x positions.")
    if not y_posns:
        raise PlotError("No clades found in the tree for y positions.")
    xmax = max(x_posns.values())
    ax.set_xlim(-0.05 * xmax, 1.05 * xmax)
    top_margin = 0.5
    ymax_plot = max(y_posns.values()) + top_margin
    ax.set_ylim(ymax_plot, -0.5)
    ax_scale = ax.get_xlim()[1] - ax.get_xlim()[0]

    get_label_color = lambda label: "black"

    if label_colors is not None:
        get_label_color = (
            label_colors
            if callable(label_colors)
            else lambda label: label_colors.get(label, "black")
        )
    else:
        clade_colors = {}
        for clade in input_tree.find_clades():
            sample = clade.name
            if sample is None:
                continue
            is_terminal = clade.is_terminal()
            clade_colors[sample] = (
                COLORS["marker_terminal"] if is_terminal else COLORS["marker_internal"]
            )
            if sample == outgroup:
                clade_colors[sample] = COLORS["marker_normal"]
        get_label_color = lambda label: clade_colors.get(label, "black")

    marker_size = marker_size if marker_size is not None else SIZES["tree_marker"]
    # Only terminal nodes are colored by cluster, internal nodes have default color
    marker_func = lambda x: (
        (marker_size, get_clade_color(x)) if x.is_terminal() else None
    )

    # Hide axes and spines
    ax.axes.get_yaxis().set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True, prune=None))
    ax.xaxis.set_tick_params(labelsize=SIZES["xlabel_tick"])
    ax.xaxis.label.set_size(SIZES["xlabel_font"])
    ax.set_title(
        title,
        x=0.01,
        y=1.0,
        ha="left",
        va="bottom",
        fontweight="bold",
        fontsize=16,
        zorder=10,
    )

    def format_branch_label(clade):
        if not branch_labels:
            if show_branch_lengths:
                return (
                    value_to_str(round(clade.branch_length, 1))
                    if clade.name != "root" and clade.name is not None
                    else None
                )
            else:
                return None
        elif isinstance(branch_labels, dict):
            return branch_labels.get(clade)
        elif callable(branch_labels):
            return value_to_str(branch_labels(clade))
        else:
            raise TypeError(
                "branch_labels must be either a dict or a callable (function)"
            )

    def value_to_str(value):
        if value is None or value == 0:
            return None
        return str(int(value)) if int(value) == value else str(value)

    if show_branch_support:

        def format_support_value(clade):
            if clade.name == "root" or clade.name is None:
                return None
            try:
                confidences = clade.confidences
            except AttributeError:
                pass
            else:
                return "/".join(value_to_str(cnf.value) for cnf in confidences)
            return (
                value_to_str(clade.confidence) if clade.confidence is not None else None
            )

    def draw_clade_lines(
        use_linecollection: bool = False,
        orientation: str = "horizontal",
        y_here: float = 0,
        x_start: float = 0,
        x_here: float = 0,
        y_bot: float = 0,
        y_top: float = 0,
        color: str = "black",
        lw: float = 0.1,
    ) -> None:
        if use_linecollection and orientation == "horizontal":
            horizontal_lines.append([(x_start, y_here), (x_here, y_here)])
            horizontal_colors.append(color)
            horizontal_lws.append(lw)
        elif use_linecollection and orientation == "vertical":
            vertical_lines.append([(x_here, y_bot), (x_here, y_top)])
            vertical_colors.append(color)
            vertical_lws.append(lw)

    def draw_clade(clade: Any, x_start: float, default_color: str, lw: float) -> None:
        x_here = x_posns[clade]
        y_here = y_posns[clade]
        color = get_clade_color(clade)

        draw_clade_lines(
            use_linecollection=True,
            orientation="horizontal",
            y_here=y_here,
            x_start=x_start,
            x_here=x_here,
            color=color,
            lw=lw,
        )

        # Markers
        if clade is not None and not (hide_internal_nodes and not clade.is_terminal()):
            marker = marker_func(clade)
            if marker is not None:
                m_size, m_color = marker
                marker_x.append(x_here)
                marker_y.append(y_here)
                marker_sizes.append(m_size)
                marker_colors.append(m_color)

        # Text labels
        label = format_branch_label(clade)
        if label not in (None, clade.__class__.__name__) and not (
            hide_internal_nodes and not clade.is_terminal()
        ):
            text_x.append(x_here + min(0.02 * ax_scale, 1))
            text_y.append(y_here)
            texts.append(f" {label}")
            text_colors.append(get_label_color(label))

        if clade.clades:
            y_top = y_posns[clade.clades[0]]
            y_bot = y_posns[clade.clades[-1]]
            draw_clade_lines(
                use_linecollection=True,
                orientation="vertical",
                x_here=x_here,
                y_bot=y_bot,
                y_top=y_top,
                color=color,
                lw=lw,
            )
            for child in clade:
                draw_clade(child, x_here, color, lw)

    line_width = (
        line_width if line_width is not None else plt.rcParams["lines.linewidth"]
    )
    draw_clade(input_tree.root, 0, "k", line_width)

    # Add line collections
    if horizontal_lines:
        h_linecollection = mpcollections.LineCollection(
            horizontal_lines,
            colors=horizontal_colors,
            linewidths=horizontal_lws,
        )
        ax.add_collection(h_linecollection)

    if vertical_lines:
        v_linecollection = mpcollections.LineCollection(
            vertical_lines,
            colors=vertical_colors,
            linewidths=vertical_lws,
        )
        ax.add_collection(v_linecollection)

    # Plot markers
    if marker_x:
        ax.scatter(marker_x, marker_y, s=marker_sizes, c=marker_colors, zorder=3)

    # Plot texts
    for xx, yy, text, color in zip(text_x, text_y, texts, text_colors):
        ax.text(xx, yy, text, verticalalignment="center", color=color)

    ax.set_xlabel("branch length")
    ax.set_ylabel("taxa")

    # Apply additional pyplot options
    for key, value in kwargs.items():
        try:
            list(value)
        except TypeError:
            raise ValueError(
                f'Keyword argument "{key}={value}" is not in the correct format.'
            ) from None
        if isinstance(value, dict):
            getattr(plt, str(key))(**dict(value))
        elif not (isinstance(value[0], tuple)):
            getattr(plt, str(key))(*value)
        elif isinstance(value[0], tuple):
            getattr(plt, str(key))(*value[0], **dict(value[1]))

    # Highlight clusters by coloring terminal nodes and drawing lines to vertical lines
    if highlight_clusters and cluster_assignments:
        # Filter cluster_assignments to only terminal nodes
        terminal_cluster_assignments = {
            cl: c_id for cl, c_id in cluster_assignments.items() if cl.is_terminal()
        }
        # Group terminal clades by cluster
        clusters = defaultdict(list)
        for clade, c_id in terminal_cluster_assignments.items():
            clusters[c_id].append(clade)

        num_clusters = len(clusters)
        if num_clusters == 0:
            print("No terminal clusters to highlight.")
        else:
            # Define spacing and starting x position for cluster lines
            x_lim = ax.get_xlim()
            y_lim = ax.get_ylim()
            margin = 0.05 * (x_lim[1] - x_lim[0])
            spacing = 0.1 * (x_lim[1] - x_lim[0])
            x_cluster_start = x_lim[1] + margin
            # Assign each cluster a unique x position
            cluster_order = sorted(clusters.keys())  # or any other order
            x_clusters = {
                c_id: x_cluster_start + i * spacing
                for i, c_id in enumerate(cluster_order)
            }

            # Adjust plot limits to accommodate cluster lines
            new_xlim = x_cluster_start + (num_clusters - 1) * spacing + margin
            ax.set_xlim(x_lim[0], new_xlim)

            for c_id, clades in clusters.items():
                x_cluster = x_clusters[c_id]
                # Draw vertical line for the cluster
                y_values = [y_posns[cl] for cl in clades]
                y_min = min(y_values)
                y_max = max(y_values)
                # If cluster has only one node, expand y_min and y_max slightly
                if y_min == y_max:
                    y_min -= 0.5
                    y_max += 0.5
                ax.plot(
                    [x_cluster, x_cluster],
                    [y_min, y_max],
                    color=cluster_colors.get(c_id, "tan"),
                    lw=2,
                )
                # Label the cluster
                y_mid = (y_min + y_max) / 2.0
                ax.text(
                    x_cluster + 0.02 * (new_xlim - x_lim[0]),
                    y_mid,
                    f"Cluster {c_id}",
                    va="center",
                    ha="left",
                    color=cluster_colors.get(c_id, "tan"),
                    fontsize=12,
                )
                # Draw lines from each terminal node to the cluster's vertical line
                for clade in clades:
                    x_node = x_posns[clade]
                    y_node = y_posns[clade]
                    ax.plot(
                        [x_node, x_cluster],
                        [y_node, y_node],
                        color=cluster_colors.get(c_id, "tan"),
                        lw=1,
                    )

    if output_name is not None:
        plt.savefig(output_name + ".png", bbox_inches="tight")

    return plt.gcf()

In [ ]:
from typing import Any, Callable, Optional, Union, Dict
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from Bio.Phylo.BaseTree import Clade
import matplotlib.collections as mpcollections
from collections import defaultdict
import pandas as pd

# Define PlotError
class PlotError(Exception):
    pass

# Define SIZES, COLORS, LINEWIDTHS
SIZES = {
    "tree_marker": 50,
    "xlabel_tick": 10,
    "xlabel_font": 12,
}
COLORS = {
    "marker_terminal": "black",
    "marker_internal": "black",
    "marker_normal": "black",
}
LINEWIDTHS = {"segment_boundary": 1.0}

def _get_x_positions(tree: Any) -> Dict[Any, float]:
    """Create a mapping of each clade to its horizontal position.
    Dict of {clade: x-coord}
    """
    depths = tree.depths()
    # If there are no branch lengths, assume unit branch lengths
    if not max(depths.values()):
        depths = tree.depths(unit_branch_lengths=True)
    return depths

def _get_y_positions(
    tree: Any, adjust: bool = False, outgroup: Optional[str] = "outgroup"
) -> Dict[Any, float]:
    """Create a mapping of each clade to its vertical position.
    Dict of {clade: y-coord}.
    Coordinates are negative, and integers for tips.
    """
    maxheight = tree.count_terminals()
    heights = {
        tip: maxheight - 1 - i
        for i, tip in enumerate(
            reversed(
                [
                    x
                    for x in tree.get_terminals()
                    if outgroup is None or x.name != outgroup
                ]
            )
        )
    }
    if outgroup is not None:
        normal_clades = list(tree.find_clades(outgroup))
        if not normal_clades:
            raise PlotError(f"Normal clade '{outgroup}' not found in tree")
        heights[normal_clades[0]] = maxheight

    # Internal nodes: place at midpoint of children
    def calc_row(clade: Any) -> None:
        for subclade in clade:
            if subclade not in heights:
                calc_row(subclade)
        # Closure over heights
        heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0

    if tree.root.clades:
        calc_row(tree.root)

    if adjust:
        pos = pd.DataFrame(
            [(clade, val) for clade, val in heights.items()], columns=["clade", "pos"]
        ).sort_values("pos")
        pos["newpos"] = 0
        count = 0
        for i in pos.index:
            if pos.loc[i, "clade"] != tree.root:
                count += 1
            pos.loc[i, "newpos"] = count

        pos.set_index("clade", inplace=True)
        heights = pos.to_dict()["newpos"]

    return heights

def plot_tree(
    input_tree: Any,
    label_func: Optional[Callable[[Any], str]] = None,
    title: str = "",
    ax: Optional[plt.Axes] = None,
    output_name: Optional[str] = None,
    outgroup: Optional[str] = None,
    width_scale: float = 1,
    height_scale: float = 1,
    show_terminal_labels: bool = False,
    show_branch_lengths: bool = False,  # Set default to False to hide branch lengths
    show_branch_support: bool = False,
    show_events: bool = False,
    branch_labels: Optional[Union[Dict[Any, str], Callable[[Any], str]]] = None,
    label_colors: Optional[Union[Dict[str, str], Callable[[str], str]]] = None,
    hide_internal_nodes: bool = False,
    marker_size: Optional[int] = None,
    line_width: Optional[float] = None,
    cluster_assignments: Optional[Dict[Clade, int]] = None,
    cluster_colors: Optional[Dict[int, str]] = None,
    highlight_clusters: bool = True,
    **kwargs: Any,
) -> plt.Figure:
    """Plot the given tree using matplotlib and highlight clusters by coloring terminal nodes and connecting lines."""

    try:
        import matplotlib.pyplot as plt
    except ImportError:
        raise ImportError("Install matplotlib or pylab if you want to use draw.")

    import matplotlib.collections as mpcollections

    marker_size = marker_size or SIZES["tree_marker"]
    line_width = line_width or LINEWIDTHS.get("segment_boundary", 1.0)
    # Default label function: return empty string if no name
    label_func = label_func or (
        lambda x: "" if hasattr(x, "name") and x.name is None else str(x)
    )

    horizontal_lines = []
    vertical_lines = []
    horizontal_colors = []
    vertical_colors = []
    horizontal_lws = []
    vertical_lws = []

    marker_x = []
    marker_y = []
    marker_sizes = []
    marker_colors = []

    text_x = []
    text_y = []
    texts = []
    text_colors = []

    # Setup figure if not given
    if ax is None:
        nsamp = len(list(input_tree.find_clades()))
        plot_height = height_scale * nsamp * 0.25
        max_leaf_to_root_distances = np.max(
            [
                np.sum([x.branch_length for x in input_tree.get_path(leaf)])
                for leaf in input_tree.get_terminals()
            ]
        )
        # Adjust plot_width based on width_scale and branch lengths
        plot_width = 5 + width_scale * max_leaf_to_root_distances
        fig, ax = plt.subplots(figsize=(min(250, plot_width), min(250, plot_height)))
    else:
        fig = plt.gcf()

    # Function to determine clade color for markers (only terminal nodes based on cluster assignments)
    def get_marker_color(clade):
        if cluster_assignments and clade in cluster_assignments and clade.is_terminal():
            c_id = cluster_assignments[clade]
            return cluster_colors.get(c_id, "tan") if cluster_colors else "tan"
        else:
            # fallback: terminal and internal nodes are black
            if clade.is_terminal():
                return COLORS["marker_terminal"]
            else:
                return COLORS["marker_internal"]

    # Scale x positions based on width_scale
    original_x_posns = _get_x_positions(input_tree)
    x_posns = {cl: x * width_scale for cl, x in original_x_posns.items()}
    y_posns = _get_y_positions(
        input_tree, adjust=not hide_internal_nodes, outgroup=outgroup
    )
    if not x_posns:
        raise PlotError("No clades found in the tree for x positions.")
    if not y_posns:
        raise PlotError("No clades found in the tree for y positions.")
    xmax = max(x_posns.values())

    # Define margin for cluster annotations
    margin = 0.1 * xmax
    # Set plot limits
    ax.set_xlim(-0.05 * xmax, xmax + margin + 0.2 * xmax)
    top_margin = 0.5
    ymax_plot = max(y_posns.values()) + top_margin
    ax.set_ylim(ymax_plot, -0.5)
    ax_scale = ax.get_xlim()[1] - ax.get_xlim()[0]

    get_label_color = lambda label: "black"

    if label_colors is not None:
        get_label_color = (
            label_colors
            if callable(label_colors)
            else lambda label: label_colors.get(label, "black")
        )
    else:
        clade_colors = {}
        for clade in input_tree.find_clades():
            sample = clade.name
            if sample is None:
                continue
            is_terminal = clade.is_terminal()
            clade_colors[sample] = (
                COLORS["marker_terminal"] if is_terminal else COLORS["marker_internal"]
            )
            if sample == outgroup:
                clade_colors[sample] = COLORS["marker_normal"]
        get_label_color = lambda label: clade_colors.get(label, "black")

    # Marker Function: Only terminal nodes are colored based on cluster assignments
    marker_func = lambda x: (
        (marker_size, get_marker_color(x)) if x.is_terminal() else None
    )

    # Hide axes and spines
    ax.axes.get_yaxis().set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True, prune=None))
    ax.xaxis.set_tick_params(labelsize=SIZES["xlabel_tick"])
    # Remove x-axis label since we'll add a scale bar instead
    ax.set_xlabel("")
    # Set title
    ax.set_title(
        title,
        x=0.01,
        y=1.0,
        ha="left",
        va="bottom",
        fontweight="bold",
        fontsize=16,
        zorder=10,
    )

    def format_branch_label(clade):
        if not branch_labels:
            if show_branch_lengths:
                return (
                    value_to_str(round(clade.branch_length, 1))
                    if clade.name != "root" and clade.name is not None
                    else None
                )
            else:
                return None
        elif isinstance(branch_labels, dict):
            return branch_labels.get(clade)
        elif callable(branch_labels):
            return value_to_str(branch_labels(clade))
        else:
            raise TypeError(
                "branch_labels must be either a dict or a callable (function)"
            )

    def value_to_str(value):
        if value is None or value == 0:
            return None
        return str(int(value)) if int(value) == value else str(value)

    if show_branch_support:
        def format_support_value(clade):
            if clade.name == "root" or clade.name is None:
                return None
            try:
                confidences = clade.confidences
            except AttributeError:
                pass
            else:
                return "/".join(value_to_str(cnf.value) for cnf in confidences)
            return (
                value_to_str(clade.confidence) if clade.confidence is not None else None
            )

    def draw_clade_lines(
        use_linecollection: bool = False,
        orientation: str = "horizontal",
        y_here: float = 0,
        x_start: float = 0,
        x_here: float = 0,
        y_bot: float = 0,
        y_top: float = 0,
        color: str = "black",
        lw: float = 0.1,
        linestyle: str = "solid",  # Added linestyle parameter
    ) -> None:
        if use_linecollection and orientation == "horizontal":
            horizontal_lines.append([(x_start, y_here), (x_here, y_here)])
            horizontal_colors.append(color)
            horizontal_lws.append(lw)
        elif use_linecollection and orientation == "vertical":
            vertical_lines.append([(x_here, y_bot), (x_here, y_top)])
            vertical_colors.append(color)
            vertical_lws.append(lw)

    def draw_clade(clade: Any, x_start: float, default_color: str, lw: float) -> None:
        x_here = x_posns[clade]
        y_here = y_posns[clade]
        # Always use black for tree branch lines
        branch_color = "black"

        draw_clade_lines(
            use_linecollection=True,
            orientation="horizontal",
            y_here=y_here,
            x_start=x_start,
            x_here=x_here,
            color=branch_color,
            lw=lw,
        )

        # Markers
        if clade is not None and not (hide_internal_nodes and not clade.is_terminal()):
            marker = marker_func(clade)
            if marker is not None:
                m_size, m_color = marker
                marker_x.append(x_here)
                marker_y.append(y_here)
                marker_sizes.append(m_size)
                marker_colors.append(m_color)

        # Text labels
        label = format_branch_label(clade)
        if label not in (None, clade.__class__.__name__) and not (
            hide_internal_nodes and not clade.is_terminal()
        ):
            text_x.append(x_here + min(0.02 * ax_scale, 1))
            text_y.append(y_here)
            texts.append(f" {label}")
            text_colors.append(get_label_color(label))

        if clade.clades:
            y_top = y_posns[clade.clades[0]]
            y_bot = y_posns[clade.clades[-1]]
            draw_clade_lines(
                use_linecollection=True,
                orientation="vertical",
                x_here=x_here,
                y_bot=y_bot,
                y_top=y_top,
                color=branch_color,  # Always black for tree branches
                lw=lw,
            )
            for child in clade:
                draw_clade(child, x_here, branch_color, lw)

    line_width = (
        line_width if line_width is not None else plt.rcParams["lines.linewidth"]
    )
    draw_clade(input_tree.root, 0, "k", line_width)

    # Add line collections
    if horizontal_lines:
        h_linecollection = mpcollections.LineCollection(
            horizontal_lines,
            colors=horizontal_colors,
            linewidths=horizontal_lws,
        )
        ax.add_collection(h_linecollection)

    if vertical_lines:
        v_linecollection = mpcollections.LineCollection(
            vertical_lines,
            colors=vertical_colors,
            linewidths=vertical_lws,
        )
        ax.add_collection(v_linecollection)

    # Plot markers
    if marker_x:
        ax.scatter(marker_x, marker_y, s=marker_sizes, c=marker_colors, zorder=3)

    # Plot texts
    for xx, yy, text, color in zip(text_x, text_y, texts, text_colors):
        ax.text(xx, yy, text, verticalalignment="center", color=color)

    # Remove the default x-axis label since we'll add a scale bar instead
    ax.set_xlabel("")
    # Remove y-axis label as it's not necessary
    ax.set_ylabel("")

    # Apply additional pyplot options
    for key, value in kwargs.items():
        try:
            list(value)
        except TypeError:
            raise ValueError(
                f'Keyword argument "{key}={value}" is not in the correct format.'
            ) from None
        if isinstance(value, dict):
            getattr(plt, str(key))(**dict(value))
        elif not (isinstance(value[0], tuple)):
            getattr(plt, str(key))(*value)
        elif isinstance(value[0], tuple):
            getattr(plt, str(key))(*value[0], **dict(value[1]))

    # Highlight clusters by coloring terminal nodes and drawing lines to a single vertical line
    if highlight_clusters and cluster_assignments:
        # Filter cluster_assignments to only terminal nodes
        terminal_cluster_assignments = {
            cl: c_id for cl, c_id in cluster_assignments.items() if cl.is_terminal()
        }
        # Group terminal clades by cluster
        clusters = defaultdict(list)
        for clade, c_id in terminal_cluster_assignments.items():
            clusters[c_id].append(clade)

        num_clusters = len(clusters)
        if num_clusters == 0:
            print("No terminal clusters to highlight.")
        else:
            # Define spacing and starting x position for cluster lines
            x_lim = ax.get_xlim()
            y_lim = ax.get_ylim()
            margin = 0.05 * (x_lim[1] - x_lim[0])
            # All clusters will share the same x_cluster position for alignment
            x_cluster = x_lim[1] + margin

            # Adjust plot limits to accommodate cluster lines and scale bar
            scale_bar_length = 10  # Define the scale bar length in branch length units
            scale_bar_position_y = y_lim[0] + 1  # Position scale bar near the bottom
            new_xlim = x_cluster + 0.2 * (x_lim[1] - x_lim[0])
            ax.set_xlim(x_lim[0], new_xlim)

            for c_id, clades in sorted(clusters.items()):
                # Collect y positions of all clades in this cluster
                y_values = [y_posns[cl] for cl in clades]
                y_min = min(y_values)
                y_max = max(y_values)
                # If cluster has only one node, expand y_min and y_max slightly
                if y_min == y_max:
                    y_min -= 0.5
                    y_max += 0.5
                # Draw vertical line for the cluster at x_cluster with increased thickness
                ax.plot(
                    [x_cluster, x_cluster],
                    [y_min, y_max],
                    color=cluster_colors.get(c_id, "tan"),
                    lw=15,  # Reasonable line width for vertical lines
                )
                # Label the cluster at the middle of the vertical line
                y_mid = (y_min + y_max) / 2.0
                ax.text(
                    x_cluster + 0.02 * (new_xlim - x_lim[0]),
                    y_mid,
                    f"Cluster {c_id}",
                    va='center',
                    ha='left',
                    color=cluster_colors.get(c_id, "tan"),
                    fontsize=12,
                )
                # Draw dotted lines from each terminal node to the cluster's vertical line
                for clade in clades:
                    x_node = x_posns[clade]
                    y_node = y_posns[clade]
                    # Only draw the line between node and cluster label, not affecting the tree branches
                    ax.plot(
                        [x_node, x_cluster],
                        [y_node, y_node],
                        color=cluster_colors.get(c_id, "tan"),
                        lw=1,
                        linestyle="dotted",  # Set connecting lines to be dotted
                    )

            # Add a scale bar
            # Define the scale bar properties
            scale_length = 10  # Length of the scale bar in branch length units
            scale_start_x = x_cluster  # Start at the same x_cluster position
            scale_start_y = y_lim[0] + 1  # Adjust y position as needed
            scale_end_x = scale_start_x + scale_length
            scale_end_y = scale_start_y

            # Draw the scale bar
            ax.plot(
                [scale_start_x, scale_end_x],
                [scale_start_y, scale_end_y],
                color='black',
                lw=2
            )
            # Label the scale bar
            ax.text(
                (scale_start_x + scale_end_x) / 2,
                scale_end_y - 0.5,  # Slightly below the scale bar
                f"{scale_length} units",
                ha='center',
                va='top',
                fontsize=12
            )

    if output_name is not None:
        plt.savefig(output_name + ".png", bbox_inches="tight")

    return plt.gcf()


In [ ]:
import matplotlib.pyplot as plt

class TreeNode:
    def __init__(self, name=None, length=0.0):
        self.name = name            # Name of the node (e.g., species name)
        self.length = length        # Branch length from parent to this node
        self.children = []          # List of child TreeNode objects
        self.parent = None          # Parent TreeNode
        self.x = 0.0                # Horizontal position for plotting
        self.y = 0.0                # Vertical position for plotting

def parse_newick(newick_str):
    """
    Parses a Newick formatted string and returns the root of the tree.
    """
    tokens = iter(newick_str.strip())

    def parse_subtree():
        node = TreeNode()
        try:
            char = next(tokens)
        except StopIteration:
            return node

        if char == '(':  # Start of a subtree
            while True:
                child = parse_subtree()
                child.parent = node
                node.children.append(child)
                try:
                    char = next(tokens)
                    if char == ',':
                        continue
                    elif char == ')':
                        break
                except StopIteration:
                    break
            # After ')', there might be a name and/or length
            name, length = parse_label_length()
            node.name = name
            node.length = length
        else:
            # It's a leaf node
            name, length = parse_label_length(initial_char=char)
            node.name = name
            node.length = length
        return node

    def parse_label_length(initial_char=None):
        """
        Parses the label and length starting from the current position.
        If initial_char is provided, it starts with that character.
        Returns a tuple (name, length).
        """
        name = initial_char if initial_char else ''
        length = ''
        # Parse name
        while True:
            try:
                char = next(tokens)
                if char == ':' or char in [',', '(', ')', ';']:
                    break
                name += char
            except StopIteration:
                break
        # Parse length
        if char == ':':
            while True:
                try:
                    char = next(tokens)
                    if char in [',', '(', ')', ';']:
                        break
                    length += char
                except StopIteration:
                    break
        else:
            # char is already one of [',', '(', ')', ';']
            # Push it back by creating a generator that yields it first
            tokens = prepend(char, tokens)
        return name.strip(), float(length) if length else 0.0

    def prepend(value, iterator):
        """
        Prepend a single value in front of an iterator.
        """
        yield value
        for item in iterator:
            yield item

    root = parse_subtree()
    return root

def assign_coordinates(node, current_x=0.0, leaf_positions=None):
    """
    Assigns x and y coordinates to each node in the tree.
    x: cumulative branch lengths from the root.
    y: assigned based on leaf order.
    """
    if leaf_positions is None:
        leaf_positions = []

    node.x = current_x
    if not node.children:
        # It's a leaf
        leaf_positions.append(node)
    else:
        for child in node.children:
            assign_coordinates(child, current_x + child.length, leaf_positions)

    return leaf_positions

def normalize_y(leaf_nodes):
    """
    Assigns y positions to leaves and propagates to internal nodes.
    Leaves are spaced evenly on the y-axis.
    Internal nodes are positioned as the average of their children's y positions.
    """
    # Assign y positions to leaves
    for i, leaf in enumerate(leaf_nodes):
        leaf.y = i

    # Propagate y positions to internal nodes
    def set_parent_y(node):
        if node.children:
            for child in node.children:
                set_parent_y(child)
            node.y = (node.children[0].y + node.children[-1].y) / 2

    # Find the root (assumes there's only one root)
    root = leaf_nodes[0]
    while root.parent:
        root = root.parent
    set_parent_y(root)

def plot_tree(node, ax):
    """
    Recursively plots the tree by drawing lines between parent and child nodes.
    """
    for child in node.children:
        # Draw a horizontal line (branch length)
        ax.plot([node.x, child.x], [node.y, child.y], color='k')
        # Recursively plot the subtree
        plot_tree(child, ax)

def add_labels(node, ax):
    """
    Recursively adds labels to the leaf nodes.
    """
    if not node.children:
        ax.text(node.x, node.y, node.name, verticalalignment='center', horizontalalignment='left')
    else:
        for child in node.children:
            add_labels(child, ax)

if __name__ == "__main__":
    # Example Newick string
    newick_str = "((A:0.1,B:0.2):0.3,(C:0.4,D:0.5):0.6);"

    # Parse the Newick string
    root = parse_newick(newick_str)

    if root is None:
        raise ValueError("Failed to parse Newick string. Please check the format.")

    # Assign coordinates
    leaf_nodes = assign_coordinates(root)
    normalize_y(leaf_nodes)

    # Create plot
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_tree(root, ax)
    add_labels(root, ax)

    # Adjust plot
    ax.invert_xaxis()  # Optional: invert x-axis to have root on left
    ax.set_xlabel('Branch Length')
    ax.set_ylabel('Leaves')
    ax.set_title('Phylogenetic Tree')
    ax.set_ylim(-1, len(leaf_nodes) + 1)  # Add some padding
    ax.set_xlim(0, max(node.x for node in leaf_nodes) + 0.1)

    # Hide y-axis ticks
    ax.get_yaxis().set_visible(False)

    plt.show()


In [ ]:
cluster_assignments = mrca_dict_1

cluster_colors = {
    0:"green",
    1: "purple",
    2: "red",
    3: "blue",
    4: "orange",
    5: "yellow",
    6: "cyan",
    7: "magenta",
    8: "black",
    9: "brown",
    10: "pink",
    11: "grey",
}

fig = plot_tree(
    to_prune, cluster_assignments=cluster_assignments, cluster_colors=cluster_colors, width_scale=0.04, height_scale=1, line_width =3, marker_size=150
)
plt.show()

# create colour map where for two clusters, if one is a subset of another the colour is a gradient based thing

In [ ]:
mrca_dict_1

In [ ]:
mrca_dict_2

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba, to_hex
import matplotlib.colors as mcolors
from collections import defaultdict


def create_cluster2_colormap(cluster1_assignments, cluster2_assignments, alpha=0.6):
    """
    Create a color map for cluster2 based on cluster1_assignments and cluster2_assignments.
    If a cluster2 is a subset of a cluster1, use cluster1's color with alpha.
    Otherwise, assign a new unique color.

    Parameters:
    - cluster1_assignments: dict {clade: cluster1_id}
    - cluster2_assignments: dict {clade: cluster2_id}
    - alpha: float, transparency for subclusters

    Returns:
    - cluster1_colors: dict {cluster1_id: color}
    - cluster2_colors: dict {cluster2_id: color}
    """
    # Step 1: Build reverse mappings
    cluster1_to_clades = defaultdict(set)
    for clade, c1_id in cluster1_assignments.items():
        cluster1_to_clades[c1_id].add(clade)

    cluster2_to_clades = defaultdict(set)
    for clade, c2_id in cluster2_assignments.items():
        cluster2_to_clades[c2_id].add(clade)

    # Step 2: Assign base colors to cluster1
    unique_c1_ids = sorted(cluster1_to_clades.keys())
    cmap1 = plt.get_cmap("tab10")  # Primary colormap for cluster1
    cluster1_colors = {}
    for idx, c1_id in enumerate(unique_c1_ids):
        cluster1_colors[c1_id] = cmap1(idx % cmap1.N)

    # Step 3: Assign colors to cluster2
    cluster2_colors = {}
    # Keep track of colors used for new clusters
    used_colors = set(to_hex(c) for c in cluster1_colors.values())
    # Use a different colormap for new clusters to avoid confusion
    cmap2 = plt.get_cmap("tab20")  # Secondary colormap for unique cluster2
    color_idx = 0

    for c2_id, clades2 in cluster2_to_clades.items():
        # Find if clades2 is a subset of any cluster1
        parent_c1_id = None
        for c1_id, clades1 in cluster1_to_clades.items():
            if clades2.issubset(clades1):
                parent_c1_id = c1_id
                break
        if parent_c1_id is not None:
            # Use cluster1's color with adjusted alpha
            base_color = cluster1_colors[parent_c1_id]
            rgba = to_rgba(base_color)
            rgba_alpha = (rgba[0], rgba[1], rgba[2], alpha)
            cluster2_colors[c2_id] = rgba_alpha
        else:
            # Assign a new unique color
            while True:
                new_color = cmap2(color_idx % cmap2.N)
                color_hex = to_hex(new_color)
                if color_hex not in used_colors:
                    break
                color_idx += 1
            cluster2_colors[c2_id] = new_color
            used_colors.add(color_hex)
            color_idx += 1
    return cluster1_colors, cluster2_colors


# Example usage
cluster2_colors = create_dynamic_cluster2_colormap(
    mrca_dict_1, mrca_dict_2,cluster_colors=cluster_colors
)
cluster1_colors = create_dynamic_cluster2_colormap(
    mrca_dict_2, mrca_dict_1, cluster_colors=cluster_colors
)
# # Print color mappings
# print("Cluster1 Colors:")
# for k, v in cluster1_colors.items():
#     print(f"Cluster1 {k}: {v}")

# print("\nCluster2 Colors:")
# for k, v in cluster2_colors.items():
#     print(f"Cluster2 {k}: {v}")

In [ ]:
fig = plot_tree(
    input_tree=to_prune,
    # title="Phylogenetic Tree with Cluster Highlights",
    cluster_assignments=mrca_dict_1,
    cluster_colors=cluster1_colors,
    width_scale=0.04,  # Adjust as needed
    show_branch_lengths=False,
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba, to_hex
import matplotlib.colors as mcolors
from collections import defaultdict
from Bio.Phylo.BaseTree import Clade
from Bio.Phylo.Newick import Tree


def create_dynamic_cluster2_colormap(
    cluster1_assignments: Dict[str, int],
    cluster2_assignments: Dict[str, int],
    cluster_colors: Dict[int, str],
    alpha_start: float = 0.8,
    alpha_step: float = 0.2,
    min_alpha: float = 0.2,
) -> Dict[int, tuple]:
    """
    Create a color map for cluster2_assignments based on cluster1_assignments and cluster_colors.
    Subclusters inherit the parent cluster's color with varying alpha based on their order.
    If a subcluster is not a subset of any primary cluster, assign it a unique color.

    Parameters:
    - cluster1_assignments: dict {clade: cluster1_id}
    - cluster2_assignments: dict {clade: cluster2_id}
    - cluster_colors: dict {cluster1_id: color_name}
    - alpha_start: float, starting alpha for the first subcluster
    - alpha_step: float, decrement in alpha for subsequent subclusters
    - min_alpha: float, minimum alpha value to prevent full transparency

    Returns:
    - cluster2_colors: dict {cluster2_id: RGBA tuple}
    """
    # Step 1: Build reverse mappings
    cluster1_to_clades = defaultdict(set)
    for clade, c1_id in cluster1_assignments.items():
        cluster1_to_clades[c1_id].add(clade)

    cluster2_to_clades = defaultdict(set)
    for clade, c2_id in cluster2_assignments.items():
        cluster2_to_clades[c2_id].add(clade)

    # Step 2: Assign colors to subclusters
    cluster2_colors = {}
    # Keep track of colors used for new clusters to avoid duplication
    used_colors = set(to_hex(c) for c in cluster_colors.values())
    # Use a different colormap for unique subclusters
    cmap2 = plt.get_cmap("tab20")  # Secondary colormap for unique cluster2
    color_idx = 0

    for c1_id, clades1 in cluster1_to_clades.items():
        # Find all cluster2_ids that are subsets of c1_id
        subclusters = [
            c2_id
            for c2_id, clades2 in cluster2_to_clades.items()
            if clades2.issubset(clades1)
        ]

        # Sort subclusters to assign alpha levels consistently
        subclusters_sorted = sorted(subclusters)

        for idx, c2_id in enumerate(subclusters_sorted):
            # Calculate alpha
            alpha = alpha_start - (idx * alpha_step)
            if alpha < min_alpha:
                alpha = min_alpha  # Prevent alpha from going below min_alpha

            # Get base color from cluster1_colors
            base_color = cluster_colors.get(
                c1_id, "tan"
            )  # Fallback to 'tan' if not found
            rgba = to_rgba(base_color)

            # Assign new color with adjusted alpha
            cluster2_colors[c2_id] = (rgba[0], rgba[1], rgba[2], alpha)

    # Step 3: Assign unique colors to cluster2_ids not subsets of any cluster1
    all_cluster2_ids = set(cluster2_assignments.values())
    subset_cluster2_ids = set(cluster2_colors.keys())
    non_subset_cluster2_ids = all_cluster2_ids - subset_cluster2_ids

    for c2_id in sorted(non_subset_cluster2_ids):
        # Assign a unique color from cmap2
        while True:
            new_color = cmap2(color_idx % cmap2.N)
            color_hex = to_hex(new_color)
            if color_hex not in used_colors:
                break
            color_idx += 1
        cluster2_colors[c2_id] = new_color
        used_colors.add(color_hex)
        color_idx += 1

    return cluster2_colors


def plot_tree(
    input_tree: Any,
    label_func: Optional[Callable[[Any], str]] = None,
    title: str = "",
    ax: Optional[plt.Axes] = None,
    output_name: Optional[str] = None,
    outgroup: Optional[str] = None,
    width_scale: float = 1,
    height_scale: float = 1,
    show_terminal_labels: bool = False,
    show_branch_lengths: bool = False,  # Set default to False to hide branch lengths
    show_branch_support: bool = False,
    show_events: bool = False,
    branch_labels: Optional[Union[Dict[Any, str], Callable[[Any], str]]] = None,
    label_colors: Optional[Union[Dict[str, str], Callable[[str], str]]] = None,
    hide_internal_nodes: bool = False,
    marker_size: Optional[int] = None,
    line_width: Optional[float] = None,
    cluster_assignments: Optional[Dict[Clade, int]] = None,
    cluster_colors: Optional[Dict[int, tuple]] = None,  # RGBA tuples
    highlight_clusters: bool = True,
    **kwargs: Any,
) -> plt.Figure:
    """Plot the given tree using matplotlib and highlight clusters by coloring terminal nodes and connecting lines."""

    try:
        import matplotlib.pyplot as plt
    except ImportError:
        raise ImportError("Install matplotlib or pylab if you want to use draw.")

    import matplotlib.collections as mpcollections

    marker_size = marker_size or 50
    line_width = line_width or 1.0
    # Default label function: return empty string if no name
    label_func = label_func or (
        lambda x: "" if hasattr(x, "name") and x.name is None else str(x)
    )

    horizontal_lines = []
    vertical_lines = []
    horizontal_colors = []
    vertical_colors = []
    horizontal_lws = []
    vertical_lws = []

    marker_x = []
    marker_y = []
    marker_sizes = []
    marker_colors = []

    text_x = []
    text_y = []
    texts = []
    text_colors = []

    # Setup figure if not given
    if ax is None:
        nsamp = len(list(input_tree.find_clades()))
        plot_height = height_scale * nsamp * 0.25
        max_leaf_to_root_distances = (
            max(
                [
                    sum(
                        [
                            x.branch_length
                            for x in input_tree.get_path(leaf)
                            if x.branch_length is not None
                        ]
                    )
                    for leaf in input_tree.get_terminals()
                ]
            )
            if any(leaf.branch_length for leaf in input_tree.get_terminals())
            else 10
        )
        # Adjust plot_width based on width_scale and branch lengths
        plot_width = 5 + width_scale * max_leaf_to_root_distances
        fig, ax = plt.subplots(figsize=(min(250, plot_width), min(250, plot_height)))
    else:
        fig = plt.gcf()

    # Function to determine clade color for markers (only terminal nodes based on cluster assignments)
    def get_marker_color(clade):
        if cluster_assignments and clade in cluster_assignments and clade.is_terminal():
            c_id = cluster_assignments[clade]
            return cluster_colors.get(c_id, "tan") if cluster_colors else "tan"
        else:
            # fallback: terminal and internal nodes are black
            if clade.is_terminal():
                return "black"
            else:
                return "black"

    # Scale x positions based on width_scale
    original_x_posns = _get_x_positions(input_tree)
    x_posns = {cl: x * width_scale for cl, x in original_x_posns.items()}
    y_posns = _get_y_positions(
        input_tree, adjust=not hide_internal_nodes, outgroup=outgroup
    )
    if not x_posns:
        raise PlotError("No clades found in the tree for x positions.")
    if not y_posns:
        raise PlotError("No clades found in the tree for y positions.")
    xmax = max(x_posns.values())

    # Define margin for cluster annotations
    margin = 0.1 * xmax
    # Set plot limits
    ax.set_xlim(-0.05 * xmax, xmax + margin + 0.2 * xmax)
    top_margin = 0.5
    ymax_plot = max(y_posns.values()) + top_margin
    ax.set_ylim(ymax_plot, -0.5)
    ax_scale = ax.get_xlim()[1] - ax.get_xlim()[0]

    get_label_color = lambda label: "black"

    if label_colors is not None:
        get_label_color = (
            label_colors
            if callable(label_colors)
            else lambda label: label_colors.get(label, "black")
        )
    else:
        clade_colors = {}
        for clade in input_tree.find_clades():
            sample = clade.name
            if sample is None:
                continue
            is_terminal = clade.is_terminal()
            clade_colors[sample] = "black" if is_terminal else "black"
            if sample == outgroup:
                clade_colors[sample] = "black"
        get_label_color = lambda label: clade_colors.get(label, "black")

    # Marker Function: Only terminal nodes are colored based on cluster assignments
    marker_func = lambda x: (
        (marker_size, get_marker_color(x)) if x.is_terminal() else None
    )

    # Hide axes and spines
    ax.axes.get_yaxis().set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.spines["bottom"].set_visible(False)  # Hide the bottom spine (x-axis line)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True, prune=None))
    ax.xaxis.set_tick_params(labelsize=10)
    # Remove x-axis label since we'll add a scale bar instead
    ax.set_xlabel("")
    # Remove y-axis label as it's not necessary
    ax.set_ylabel("")
    # Set title
    ax.set_title(
        title,
        x=0.01,
        y=1.0,
        ha="left",
        va="bottom",
        fontweight="bold",
        fontsize=16,
        zorder=10,
    )

    def format_branch_label(clade):
        if not branch_labels:
            if show_branch_lengths and clade.branch_length is not None:
                return f"{clade.branch_length:.1f}"
            else:
                return None
        elif isinstance(branch_labels, dict):
            return branch_labels.get(clade)
        elif callable(branch_labels):
            return branch_labels(clade)
        else:
            raise TypeError(
                "branch_labels must be either a dict or a callable (function)"
            )

    def value_to_str(value):
        if value is None or value == 0:
            return None
        return str(int(value)) if int(value) == value else str(value)

    if show_branch_support:

        def format_support_value(clade):
            if clade.name == "root" or clade.name is None:
                return None
            try:
                confidences = clade.confidences
            except AttributeError:
                pass
            else:
                return "/".join(value_to_str(cnf.value) for cnf in confidences)
            return (
                value_to_str(clade.confidence) if clade.confidence is not None else None
            )

    def draw_clade_lines(
        use_linecollection: bool = False,
        orientation: str = "horizontal",
        y_here: float = 0,
        x_start: float = 0,
        x_here: float = 0,
        y_bot: float = 0,
        y_top: float = 0,
        color: str = "black",
        lw: float = 0.1,
        linestyle: str = "solid",  # Added linestyle parameter
    ) -> None:
        if use_linecollection and orientation == "horizontal":
            horizontal_lines.append([(x_start, y_here), (x_here, y_here)])
            horizontal_colors.append(color)
            horizontal_lws.append(lw)
        elif use_linecollection and orientation == "vertical":
            vertical_lines.append([(x_here, y_bot), (x_here, y_top)])
            vertical_colors.append(color)
            vertical_lws.append(lw)

    def draw_clade(clade: Any, x_start: float, default_color: str, lw: float) -> None:
        x_here = x_posns[clade]
        y_here = y_posns[clade]
        # Always use black for tree branch lines
        branch_color = "black"

        draw_clade_lines(
            use_linecollection=True,
            orientation="horizontal",
            y_here=y_here,
            x_start=x_start,
            x_here=x_here,
            color=branch_color,
            lw=lw,
        )

        # Markers
        if clade is not None and not (hide_internal_nodes and not clade.is_terminal()):
            marker = marker_func(clade)
            if marker is not None:
                m_size, m_color = marker
                marker_x.append(x_here)
                marker_y.append(y_here)
                marker_sizes.append(m_size)
                marker_colors.append(m_color)

        # Text labels
        label = format_branch_label(clade)
        if label not in (None, clade.__class__.__name__) and not (
            hide_internal_nodes and not clade.is_terminal()
        ):
            text_x.append(x_here + min(0.02 * ax_scale, 1))
            text_y.append(y_here)
            texts.append(f" {label}")
            text_colors.append(get_label_color(label))

        if clade.clades:
            y_top = y_posns[clade.clades[0]]
            y_bot = y_posns[clade.clades[-1]]
            draw_clade_lines(
                use_linecollection=True,
                orientation="vertical",
                x_here=x_here,
                y_bot=y_bot,
                y_top=y_top,
                color=branch_color,  # Always black for tree branches
                lw=lw,
            )
            for child in clade:
                draw_clade(child, x_here, branch_color, lw)

    draw_clade(input_tree.root, 0, "k", line_width)

    # Add line collections
    if horizontal_lines:
        h_linecollection = mpcollections.LineCollection(
            horizontal_lines,
            colors=horizontal_colors,
            linewidths=horizontal_lws,
        )
        ax.add_collection(h_linecollection)

    if vertical_lines:
        v_linecollection = mpcollections.LineCollection(
            vertical_lines,
            colors=vertical_colors,
            linewidths=vertical_lws,
        )
        ax.add_collection(v_linecollection)

    # Plot markers
    if marker_x:
        ax.scatter(marker_x, marker_y, s=marker_sizes, c=marker_colors, zorder=3)

    # Plot texts
    for xx, yy, text, color in zip(text_x, text_y, texts, text_colors):
        ax.text(xx, yy, text, verticalalignment="center", color=color)

    # Remove the default x-axis label since we'll add a scale bar instead
    ax.set_xlabel("")
    # Remove y-axis label as it's not necessary
    ax.set_ylabel("")

    # Apply additional pyplot options
    for key, value in kwargs.items():
        try:
            list(value)
        except TypeError:
            raise ValueError(
                f'Keyword argument "{key}={value}" is not in the correct format.'
            ) from None
        if isinstance(value, dict):
            getattr(plt, str(key))(**dict(value))
        elif not (isinstance(value[0], tuple)):
            getattr(plt, str(key))(*value)
        elif isinstance(value[0], tuple):
            getattr(plt, str(key))(*value[0], **dict(value[1]))

    # Highlight clusters by coloring terminal nodes and drawing lines to a single vertical line
    if highlight_clusters and cluster_assignments:
        # Filter cluster_assignments to only terminal nodes
        terminal_cluster_assignments = {
            cl: c_id for cl, c_id in cluster_assignments.items() if cl.is_terminal()
        }
        # Group terminal clades by cluster
        clusters = defaultdict(list)
        for clade, c_id in terminal_cluster_assignments.items():
            clusters[c_id].append(clade)

        num_clusters = len(clusters)
        if num_clusters == 0:
            print("No terminal clusters to highlight.")
        else:
            # Define spacing and starting x position for cluster lines
            x_lim = ax.get_xlim()
            y_lim = ax.get_ylim()
            margin = 0.05 * (x_lim[1] - x_lim[0])
            # All clusters will share the same x_cluster position for alignment
            x_cluster = x_lim[1] + margin

            # Assign all clusters to the same x_cluster position to align labels vertically
            # Adjust plot limits to accommodate cluster lines
            new_xlim = x_cluster + 0.2 * (x_lim[1] - x_lim[0])
            ax.set_xlim(x_lim[0], new_xlim)

            for c_id, clades in sorted(clusters.items()):
                # Collect y positions of all clades in this cluster
                y_values = [y_posns[cl] for cl in clades]
                y_min = min(y_values)
                y_max = max(y_values)
                # If cluster has only one node, expand y_min and y_max slightly
                if y_min == y_max:
                    y_min -= 0.5
                    y_max += 0.5
                # Draw vertical line for the cluster at x_cluster with increased thickness
                ax.plot(
                    [x_cluster, x_cluster],
                    [y_min, y_max],
                    color=cluster_colors.get(c_id, "tan"),
                    lw=3,  # Reasonable line width for vertical lines
                )
                # Label the cluster at the middle of the vertical line
                y_mid = (y_min + y_max) / 2.0
                ax.text(
                    x_cluster + 0.02 * (new_xlim - x_lim[0]),
                    y_mid,
                    f"Cluster {c_id}",
                    va="center",
                    ha="left",
                    color=cluster_colors.get(c_id, "tan"),
                    fontsize=12,
                )
                # Draw dotted lines from each terminal node to the cluster's vertical line
                for clade in clades:
                    x_node = x_posns[clade]
                    y_node = y_posns[clade]
                    # Only draw the line between node and cluster label, not affecting the tree branches
                    ax.plot(
                        [x_node, x_cluster],
                        [y_node, y_node],
                        color=cluster_colors.get(c_id, "tan"),
                        lw=1,
                        linestyle="dotted",  # Set connecting lines to be dotted
                    )

    if output_name is not None:
        plt.savefig(output_name + ".png", bbox_inches="tight")

    return plt.gcf()


def _get_x_positions(tree: Any) -> Dict[Any, float]:
    """Create a mapping of each clade to its horizontal position.
    Dict of {clade: x-coord}
    """
    depths = tree.depths()
    # If there are no branch lengths, assume unit branch lengths
    if not max(depths.values()):
        depths = tree.depths(unit_branch_lengths=True)
    return depths


def _get_y_positions(
    tree: Any, adjust: bool = False, outgroup: Optional[str] = "outgroup"
) -> Dict[Any, float]:
    """Create a mapping of each clade to its vertical position.
    Dict of {clade: y-coord}.
    Coordinates are negative, and integers for tips.
    """
    maxheight = tree.count_terminals()
    heights = {
        tip: maxheight - 1 - i
        for i, tip in enumerate(
            reversed(
                [
                    x
                    for x in tree.get_terminals()
                    if outgroup is None or x.name != outgroup
                ]
            )
        )
    }
    if outgroup is not None:
        normal_clades = list(tree.find_clades(outgroup))
        if not normal_clades:
            raise PlotError(f"Normal clade '{outgroup}' not found in tree")
        heights[normal_clades[0]] = maxheight

    # Internal nodes: place at midpoint of children
    def calc_row(clade: Any) -> None:
        for subclade in clade:
            if subclade not in heights:
                calc_row(subclade)
        # Closure over heights
        heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0

    if tree.root.clades:
        calc_row(tree.root)

    if adjust:
        pos = pd.DataFrame(
            [(clade, val) for clade, val in heights.items()], columns=["clade", "pos"]
        ).sort_values("pos")
        pos["newpos"] = 0
        count = 0
        for i in pos.index:
            if pos.loc[i, "clade"] != tree.root:
                count += 1
            pos.loc[i, "newpos"] = count

        pos.set_index("clade", inplace=True)
        heights = pos.to_dict()["newpos"]

    return heights


In [ ]:
from Bio import Phylo

# Define the input and output file paths
input_file = "/home/ganesank/project/phytclust/Data/Species/Avian_time_tree.nexus"
output_file = "/home/ganesank/project/phytclust/Data/Species/Avian_time_tree.newick"

# Read the NEXUS file
tree = Phylo.read(input_file, "nexus")

Phylo.write(tree, output_file, "newick")

In [ ]:
import matplotlib.pyplot as plt


class TreeNode:
    def __init__(self, name=None, length=0.0):
        self.name = name  # Name of the node (e.g., species name)
        self.length = length  # Branch length from parent to this node
        self.children = []  # List of child TreeNode objects
        self.parent = None  # Parent TreeNode
        self.x = 0.0  # Horizontal position for plotting
        self.y = 0.0  # Vertical position for plotting


def parse_newick(newick_str):
    """
    Parses a Newick formatted string and returns the root of the tree.
    """
    tokens = iter(newick_str.strip())

    def parse_subtree():
        node = TreeNode()
        try:
            char = next(tokens)
        except StopIteration:
            return node

        if char == "(":  # Start of a subtree
            while True:
                child = parse_subtree()
                child.parent = node
                node.children.append(child)
                try:
                    char = next(tokens)
                    if char == ",":
                        continue
                    elif char == ")":
                        break
                except StopIteration:
                    break
            # After ')', there might be a name and/or length
            name, length = parse_label_length()
            node.name = name
            node.length = length
        else:
            # It's a leaf node
            name, length = parse_label_length(initial_char=char)
            node.name = name
            node.length = length
        return node

    def parse_label_length(initial_char=None):
        """
        Parses the label and length starting from the current position.
        If initial_char is provided, it starts with that character.
        Returns a tuple (name, length).
        """
        name = initial_char if initial_char else ""
        length = ""
        # Parse name
        while True:
            try:
                char = next(tokens)
                if char == ":" or char in [",", "(", ")", ";"]:
                    break
                name += char
            except StopIteration:
                break
        # Parse length
        if char == ":":
            while True:
                try:
                    char = next(tokens)
                    if char in [",", "(", ")", ";"]:
                        break
                    length += char
                except StopIteration:
                    break
        else:
            # char is already one of [',', '(', ')', ';']
            # Push it back by creating a generator that yields it first
            tokens = prepend(char, tokens)
        return name.strip(), float(length) if length else 0.0

    def prepend(value, iterator):
        """
        Prepend a single value in front of an iterator.
        """
        yield value
        for item in iterator:
            yield item

    root = parse_subtree()
    return root


def assign_coordinates(node, current_x=0.0, leaf_positions=None):
    """
    Assigns x and y coordinates to each node in the tree.
    x: cumulative branch lengths from the root.
    y: assigned based on leaf order.
    """
    if leaf_positions is None:
        leaf_positions = []

    node.x = current_x
    if not node.children:
        # It's a leaf
        leaf_positions.append(node)
    else:
        for child in node.children:
            assign_coordinates(child, current_x + child.length, leaf_positions)

    return leaf_positions


def normalize_y(leaf_nodes):
    """
    Assigns y positions to leaves and propagates to internal nodes.
    Leaves are spaced evenly on the y-axis.
    Internal nodes are positioned as the average of their children's y positions.
    """
    # Assign y positions to leaves
    for i, leaf in enumerate(leaf_nodes):
        leaf.y = i

    # Propagate y positions to internal nodes
    def set_parent_y(node):
        if node.children:
            for child in node.children:
                set_parent_y(child)
            node.y = (node.children[0].y + node.children[-1].y) / 2

    # Find the root (assumes there's only one root)
    root = leaf_nodes[0]
    while root.parent:
        root = root.parent
    set_parent_y(root)


def plot_tree(node, ax):
    """
    Recursively plots the tree by drawing lines between parent and child nodes.
    """
    for child in node.children:
        # Draw a horizontal line (branch length)
        ax.plot([node.x, child.x], [node.y, child.y], color="k")
        # Recursively plot the subtree
        plot_tree(child, ax)


def add_labels(node, ax):
    """
    Recursively adds labels to the leaf nodes.
    """
    if not node.children:
        ax.text(
            node.x,
            node.y,
            node.name,
            verticalalignment="center",
            horizontalalignment="left",
        )
    else:
        for child in node.children:
            add_labels(child, ax)


if __name__ == "__main__":
    # Example Newick string
    newick_str = "((A:0.1,B:0.2):0.3,(C:0.4,D:0.5):0.6);"

    # Parse the Newick string
    root = parse_newick(newick_str)

    if root is None:
        raise ValueError("Failed to parse Newick string. Please check the format.")

    # Assign coordinates
    leaf_nodes = assign_coordinates(root)
    normalize_y(leaf_nodes)

    # Create plot
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_tree(root, ax)
    add_labels(root, ax)

    # Adjust plot
    ax.invert_xaxis()  # Optional: invert x-axis to have root on left
    ax.set_xlabel("Branch Length")
    ax.set_ylabel("Leaves")
    ax.set_title("Phylogenetic Tree")
    ax.set_ylim(-1, len(leaf_nodes) + 1)  # Add some padding
    ax.set_xlim(0, max(node.x for node in leaf_nodes) + 0.1)

    # Hide y-axis ticks
    ax.get_yaxis().set_visible(False)

    plt.show()